# CHARGEMENT DES PACKAGES NÉCESSAIRES

In [1]:
import io
import pandas as pd
import boto3
import numpy as np
import unicodedata
import re
from typing import Optional, Dict, Tuple, List
from IPython.display import display, HTML
from datetime import datetime
import os



# FONCTIONS UTILITAIRES COMMUNES

1. `_norm_txt(x)`
Normalise un texte pour le rendre comparable quelle que soit la saisie.

**Règles appliquées :**
- Passage en minuscules (`lower()`).
- Suppression des accents via `unicodedata.normalize`.
- Remplacement des séparateurs (`- _ ' / .`) par un espace.
- Réduction des espaces multiples en un seul.
- Gestion des valeurs vides / NaN.

**Exemple :**
- `"Célibataire"` → `"celibataire"`
- `"Marié(e)"` → `"marie(e)"`

Utilisé pour tous les recodages textuels (mariage, emploi, libellés administratifs…).


2. `_sex_norm(s)`
Harmonise les valeurs de sexe pour obtenir uniquement `{M, F, NaN}`.

**Transformations :**
- Conversion en majuscules et stripping.
- Recodage :
  - `"MASCULIN"`, `"HOMME"`, `"H"` → `M`
  - `"FEMININ"`, `"FEMME"` → `F`
- Toute autre valeur devient `NaN`.

Permet une analyse propre du sexe (agrégations, imputations, contrôles).


3. `_to_datetime_mixed(s)`
Convertit une série en `datetime`, même si les formats sont mixtes.

**Fonctionnalités :**
- Priorité au format français (jour/mois/année).
- Essaie d’interpréter des formats hétérogènes dans la même colonne.
- Les valeurs non convertibles deviennent `NaT`.

Utile pour des colonnes avec des dates issues de sources multiples (Excel, texte, CSV…).


4. `_clean_dates_generic(series)`
Nettoie une colonne de dates très hétérogène.

**Cas pris en charge :**
- Heures seules (`"12:30"`), dates nulles (`"00/00/0000"`), chaînes vides → `NaT`.
- Conversion standard en datetime via `_to_datetime_mixed`.
- Détection et conversion des **numéros de série Excel** (1–60000 → 1899–12–30 + n jours).

**Objectif :**
Rendre toutes les dates interprétables pour les calculs temporels (âge, ancienneté, variations annuelles…).


5. `_mode_safe(s)`
Renvoie le **mode** d’une série (valeur la plus fréquente) de façon robuste.

**Comportement :**
- Supprime les `NaN`,
- Si la série est vide → `pd.NA`,
- Sinon renvoie le premier mode.

Utilisé pour stabiliser une variable au niveau du matricule (ex. âge stabilisé).


6. `_ensure_date_ref(df)`
Garantit l’existence et la cohérence de `DATE_REF`, date de **début de mois** servant de repère temporel unique.

**Hiérarchie utilisée :**
1. `DATE_COLLECTE` existe → convertie en début de mois.
2. Sinon, `DATE_REF` existe déjà → normalisation.
3. Sinon, colonnes ANNEE/MOIS → construction `YYYY-MM-01`.
4. Sinon → erreur.

**Pourquoi `DATE_REF` ?**
Elle sert d’axe chronologique pour :
- détecter les variations d’âge,
- recalculer l’ancienneté,
- mesurer les dérives temporelles.


7. `_need_keys(df)`
Vérifie la présence des colonnes essentielles avant tout contrôle temporel.

**Clés requises :**
- `MATRICULE`
- `DATE_REF`

Sinon, lève une `ValueError`.

Garantit que les fonctions downstream peuvent grouper correctement les trajectoires.


8. `_add_year_delta(g)`
Calcule la variation en années entre deux collectes consécutives pour un même matricule.

**Colonnes ajoutées :**
- `_DATE_PREV` : date précédente (via `shift(1)`).
- `_YEAR_DELTA` : différence en années (jours / 365.25).

**Exemple :**
- 2020-01-01 → 0.0 an  
- 2020-07-01 → 0.5 an  
- 2021-01-01 → 0.5 an

Permet d'évaluer la vitesse d’évolution de l’âge ou de l’ancienneté.


9. `CFG` – Configuration globale des seuils
Dictionnaire centralisant les tolérances utilisées dans tous les contrôles de cohérence.

**Contenu :**
- `age_bounds = (18, 75)` → âge plausible.
- `age_ps_bounds = (14, 60)` → âge plausible à la prise de service.
- `retraite_min_age = 50` → âge minimal crédible pour une retraite.
- `max_age_drift_years = 1.0` → tolérance de variation d’âge entre deux collectes.
- `max_anc_drift_years = 0.25` → variation maximale de l’ancienneté (~3 mois).

Ces paramètres sont appelés par les fonctions de contrôle pour générer les flags.


In [2]:
def _norm_txt(x):
    """
    Normalise le texte : minuscules, sans accents, espaces uniques
    
    Args:
        x: Texte à normaliser (str ou valeur pandas)
    
    Returns:
        Texte normalisé
    
    Exemple:
        "Célibataire" → "celibataire"
        "Marié(e)" → "marie(e)"
    """
    if pd.isna(x):
        return x
    x = str(x).strip().lower()
    # Supprimer les accents
    x = ''.join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))
    # Remplacer les séparateurs par des espaces
    x = re.sub(r"[-_'/\.]+", " ", x)
    # Réduire les espaces multiples en un seul
    x = re.sub(r"\s+", " ", x).strip()
    return x


def _sex_norm(s: pd.Series) -> pd.Series:
    """
    Normalise le sexe pour obtenir uniquement {M, F, NaN}
    
    Args:
        s: Série pandas contenant les valeurs de sexe
    
    Returns:
        Série normalisée avec M, F ou NaN
    
    Transformations:
        "MASCULIN", "HOMME", "H" → "M"
        "FEMININ", "FEMME" → "F"
        Autres → NaN
    """
    s = s.astype(str).str.strip().str.upper().replace({
        'MASCULIN': 'M', 'HOMME': 'M', 'H': 'M',
        'FEMININ': 'F', 'FEMME': 'F'
    })
    return s.where(s.isin(['M', 'F']))


def _to_datetime_mixed(s):
    """
    Convertit en datetime avec gestion des formats mixtes
    
    Args:
        s: Série pandas contenant des dates
    
    Returns:
        Série datetime
    
    Gère:
        - Format français (jour/mois/année)
        - Formats mixtes dans la même colonne
        - Erreurs de conversion (→ NaT)
    """
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)


def _clean_dates_generic(series):
    """
    Nettoie les dates : gère les formats variés, dates nulles, nombres série Excel
    
    Args:
        series: Série pandas contenant des dates
    
    Returns:
        Série datetime nettoyée
    
    Gère les cas particuliers:
        - Heures seules (12:30) → NaT
        - Dates nulles (00/00/0000) → NaT
        - Chaînes vides → NaT
        - Nombres Excel (44197 = 2021-01-01) → Conversion automatique
    """
    s = series.copy()
    s_str = s.astype(str).str.strip().str.lower()
    
    # Identifier les valeurs à ignorer
    time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
    zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
    empties = s_str.isin(["", "nan", "nat", "none", "nul", "null"])
    
    # Remplacer par NaN
    s = s.mask(time_only | zero_date | empties, np.nan)
    
    # Conversion datetime standard
    dt = _to_datetime_mixed(s)
    
    # Gérer les nombres série Excel (1 à 60000 = dates entre 1900 et 2064)
    serial = pd.to_numeric(s_str, errors="coerce")
    serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
    if serial_mask.any():
        dt_serial = pd.to_datetime(serial[serial_mask], unit="D", origin="1899-12-30", errors="coerce")
        dt.loc[serial_mask] = dt_serial
    
    return dt


def _mode_safe(s: pd.Series):
    """
    Retourne le mode d'une série, ou NA si vide
    
    Args:
        s: Série pandas
    
    Returns:
        Mode de la série (valeur la plus fréquente) ou NA
    """
    s_nonan = s.dropna()
    if s_nonan.empty:
        return pd.NA
    m = s_nonan.mode()
    return m.iloc[0] if not m.empty else pd.NA


def _ensure_date_ref(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assure l'existence de DATE_REF (début de mois) avec priorité à DATE_COLLECTE
    
    Args:
        df: DataFrame contenant les données
    
    Returns:
        DataFrame avec colonne DATE_REF ajoutée
    
    Stratégie en cascade:
        1. Si DATE_COLLECTE existe → convertir en début de mois
        2. Si DATE_REF existe déjà → la normaliser
        3. Si ANNEE_COLLECTE et MOIS_COLLECTE existent → combiner
        4. Sinon → erreur
    """
    df = df.copy()
    
    # Cas 1 : DATE_COLLECTE disponible
    if 'DATE_COLLECTE' in df.columns:
        s = df['DATE_COLLECTE']
        parsed_std = pd.to_datetime(s, errors='coerce')
        parsed_final = parsed_std
        
        # Détecter si format Excel (beaucoup de NaN après conversion standard)
        try:
            needs_excel = parsed_std.isna().mean() > 0.5 and pd.api.types.is_numeric_dtype(s)
        except Exception:
            needs_excel = False
            
        if needs_excel:
            parsed_excel = pd.to_datetime(s, unit='D', origin='1899-12-30', errors='coerce')
            if parsed_excel.notna().sum() > parsed_std.notna().sum():
                parsed_final = parsed_excel
        
        # Convertir en début de mois (2020-03-15 → 2020-03-01)
        df['DATE_REF'] = parsed_final.dt.to_period('M').dt.to_timestamp(how='start')
        return df
    
    # Cas 2 : DATE_REF existe déjà
    if 'DATE_REF' in df.columns:
        df['DATE_REF'] = pd.to_datetime(df['DATE_REF'], errors='coerce').dt.to_period('M').dt.to_timestamp(how='start')
        return df
    
    # Cas 3 : ANNEE et MOIS séparés
    year_candidates = ['ANNEE_COLLECTE', 'ANNEE', 'YEAR', 'AN']
    month_candidates = ['MOIS_COLLECTE', 'MOIS', 'MONTH']
    
    year_col = next((c for c in year_candidates if c in df.columns), None)
    month_col = next((c for c in month_candidates if c in df.columns), None)
    
    if year_col is not None:
        y = pd.to_numeric(df[year_col], errors='coerce').round(0).astype('Int64').astype(str)
        m = pd.to_numeric(df[month_col], errors='coerce') if month_col in df.columns else 1
        m = pd.Series(m, index=df.index).round(0).astype('Int64').astype(str).str.zfill(2)
        df['DATE_REF'] = pd.to_datetime(y + '-' + m + '-01', format='%Y-%m-%d', errors='coerce')
        return df
    
    # Cas 4 : Aucune colonne de date trouvée
    raise ValueError("DATE_REF introuvable.")


def _need_keys(df):
    """
    Vérifie la présence des clés temporelles nécessaires
    
    Args:
        df: DataFrame à vérifier
    
    Raises:
        ValueError: Si MATRICULE ou DATE_REF manquent
    """
    if 'MATRICULE' not in df.columns:
        raise ValueError("Colonne MATRICULE manquante.")
    if 'DATE_REF' not in df.columns:
        raise ValueError("Colonne DATE_REF manquante.")


def _add_year_delta(g: pd.DataFrame) -> pd.DataFrame:
    """
    Ajoute le delta temps (années) dans un groupe
    
    Args:
        g: Groupe de lignes d'un même matricule (trié par DATE_REF)
    
    Returns:
        Groupe avec colonnes _DATE_PREV et _YEAR_DELTA ajoutées
    
    Exemple:
        DATE_REF      → _DATE_PREV    → _YEAR_DELTA
        2020-01-01   →  NaT          →  0.0
        2020-07-01   →  2020-01-01   →  0.5 (6 mois)
        2021-01-01   →  2020-07-01   →  0.5 (6 mois)
    """
    g = g.sort_values('DATE_REF').copy()
    g['_DATE_PREV'] = g['DATE_REF'].shift(1)
    g['_YEAR_DELTA'] = ((g['DATE_REF'] - g['_DATE_PREV']).dt.days / 365.25).fillna(0)
    return g


# Configuration globale des seuils de validation
CFG = {
    'age_bounds': (18, 75),          # Âge valide entre 18 et 75 ans
    'age_ps_bounds': (14, 60),       # Âge de prise de service entre 14 et 60 ans
    'retraite_min_age': 50,          # Âge minimum de départ à la retraite
    'max_age_drift_years': 1.0,      # Tolérance 1 an pour variation d'âge
    'max_anc_drift_years': 0.25,     # Tolérance 3 mois pour variation d'ancienneté
}


# SYSTÈME DE CONTRÔLE ET EXPORT


**Objectif général**

Cette classe centralise l’ensemble des **contrôles de cohérence**, la **détection des flags**, et l’**export détaillé** des anomalies.  
Elle inclut une **correction majeure** :  
au lieu d’exporter uniquement les lignes problématiques,  
on exporte **toutes les observations des matricules concernés**, pour disposer d’un **contexte complet** (avant / pendant / après l’anomalie).

---

Fonctionnalités principales
1. **Analyser une variable** et afficher sa distribution.  
2. **Identifier les flags actifs** (obs. problématiques).  
3. **Lister tous les matricules concernés** par chaque flag.  
4. **Exporter toutes les observations** de ces matricules, triées chronologiquement.  
5. **Générer un rapport global consolidé** (Excel).

---

Attributs internes
- `output_dir` : répertoire où seront stockés les exports.  
- `timestamp` : horodatage automatique pour nommer les fichiers.  
- `problemes_detectes` : liste cumulant les anomalies détectées par variable.  
- `max_excel_rows` : limite au-delà de laquelle on bascule en CSV/Parquet.

---

Méthode `__init__(...)`
Initialise le module de contrôle.

**Actions :**
- Création du répertoire d’export si nécessaire.
- Génération d’un horodatage unique (`YYYYMMDD_HHMMSS`).
- Initialisation de la liste des problèmes.
- Paramétrage de la limite Excel (1 000 000 lignes par défaut).

---

Méthode `afficher_titre(titre)`
Affiche un titre formaté dans la console pour bien séparer les blocs de contrôle.

---

Méthode `controler_variable(df, nom_variable, flags_config)`
Contrôle une variable et exporte les anomalies **avec contexte complet**.

Étape 1 — Affichage du titre + distribution
Si la variable est présente :
- Génération d’un tableau Effectif / Pourcentage (%).
- Affichage structuré via `display()`.

Étape 2 — Parcours des flags configurés
Le paramètre `flags_config` est un dictionnaire :

```python
{
    'FLAG_XXX': {
        'description': "...",
        'colonnes_export': ['col1', 'col2']
    }
}


In [3]:
# Cette partie contient la classe ControleQualite avec la CORRECTION
# pour exporter TOUT le contexte des matricules problématiques.
#
# CORRECTION MAJEURE :
# - Au lieu d'exporter uniquement les lignes flaggées
# - On exporte TOUTES les observations des matricules concernés
# - Permet de voir l'historique complet avant/après le problème
# ============================================================

class ControleQualite:
    """
    Gestion des contrôles de qualité avec export Excel/CSV
    
    Cette classe permet de :
    1. Contrôler la qualité de chaque variable
    2. Détecter les problèmes via des flags
    3. Exporter les observations problématiques avec TOUT leur contexte
    4. Générer un rapport global consolidé
    
    Attributs:
        output_dir: Répertoire de sortie des fichiers
        timestamp: Horodatage pour nommer les fichiers
        problemes_detectes: Liste des problèmes trouvés
        max_excel_rows: Limite de lignes pour Excel (au-delà → CSV)
    """
    
    def __init__(self, output_dir="controles_qualite", max_excel_rows=1000000):
        """
        Initialise le système de contrôle qualité
        
        Args:
            output_dir: Dossier où sauvegarder les rapports
            max_excel_rows: Nombre max de lignes pour Excel (défaut: 1M)
        """
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.problemes_detectes = []
        self.max_excel_rows = max_excel_rows
        
    def afficher_titre(self, titre):
        """
        Affiche un titre formaté dans la console
        
        Args:
            titre: Texte du titre
        """
        print("\n" + "="*80)
        print(f"  {titre}")
        print("="*80 + "\n")
        
    def controler_variable(self, df, nom_variable, flags_config):
        """
        Contrôle une variable et exporte les problèmes - VERSION CORRIGÉE
        
        ⚠️ CORRECTION MAJEURE : Au lieu d'exporter UNIQUEMENT les lignes flaggées,
        on exporte TOUTES les observations des matricules concernés pour voir
        le contexte complet (avant/après le problème).
        
        Args:
            df: DataFrame complet
            nom_variable: Nom de la variable contrôlée
            flags_config: Dict avec structure {
                'flag_name': {
                    'description': str,
                    'colonnes_export': list
                }
            }
        
        Exemple d'utilisation:
            controle.controler_variable(
                df,
                "AGE_IMPUTE",
                {
                    'FLAG_AGE_NEGATIF': {
                        'description': 'Âge négatif détecté',
                        'colonnes_export': ['AGE', 'DATE_NAISSANCE']
                    }
                }
            )
        """
        self.afficher_titre(f"CONTRÔLE : {nom_variable}")
        
        # Afficher la distribution de la variable
        if nom_variable in df.columns:
            print(f"📊 Distribution de {nom_variable}:")
            dist = df[nom_variable].value_counts(dropna=False).sort_index()
            dist_pct = df[nom_variable].value_counts(normalize=True, dropna=False).sort_index() * 100
            
            tableau = pd.DataFrame({
                'Effectif': dist,
                'Pourcentage (%)': dist_pct.round(2)
            })
            display(tableau)
            print()
        
        # Vérifier chaque flag
        problemes_trouves = False
        
        for flag_name, config in flags_config.items():
            if flag_name not in df.columns:
                continue
                
            nb_problemes = df[flag_name].sum()
            
            if nb_problemes > 0:
                problemes_trouves = True
                
                # ============================================================
                # ✅ CORRECTION : Exporter TOUTES les lignes des matricules concernés
                # ============================================================
                
                # Étape 1 : Identifier les matricules qui ont au moins un problème
                matricules_problematiques = df[df[flag_name] == 1]['MATRICULE'].unique()
                
                # Étape 2 : Extraire TOUTES les observations de ces matricules
                # (pas seulement celles avec flag=1)
                df_probleme = df[df['MATRICULE'].isin(matricules_problematiques)].copy()
                
                # Étape 3 : Trier pour faciliter la lecture (par matricule puis par date)
                df_probleme = df_probleme.sort_values(['MATRICULE', 'DATE_REF'])
                
                # ============================================================
                # Calculer les statistiques
                nb_observations_flagees = df[df[flag_name] == 1].shape[0]  # Lignes réellement flaggées
                nb_observations_totales = len(df_probleme)                  # Lignes exportées (avec contexte)
                nb_matricules = len(matricules_problematiques)
                
                pct_obs_flagees = (nb_observations_flagees / len(df)) * 100
                pct_mat = (nb_matricules / df['MATRICULE'].nunique()) * 100
                
                print(f"⚠️  {config['description']}")
                print(f"   Observations FLAGÉES : {nb_observations_flagees:,} ({pct_obs_flagees:.2f}%)")
                print(f"   Observations exportées (avec contexte) : {nb_observations_totales:,}")
                print(f"   Nombre de matricules concernés : {nb_matricules:,} ({pct_mat:.2f}%)")
                
                # Colonnes à exporter
                colonnes_base = ['MATRICULE', 'DATE_REF', 'PERIODE', 'DATE_COLLECTE']
                colonnes_export = colonnes_base + config['colonnes_export']
                
                # ✅ Ajouter le flag pour voir quelles lignes sont problématiques
                if flag_name not in colonnes_export:
                    colonnes_export.append(flag_name)
                
                # Garder seulement les colonnes qui existent
                colonnes_export = [c for c in colonnes_export if c in df_probleme.columns]
                
                df_export = df_probleme[colonnes_export].copy()
                df_export['DESCRIPTION_PROBLEME'] = config['description']
                
                # Afficher un échantillon (10 lignes au lieu de 5)
                print(f"\n   📋 Échantillon (10 premières lignes avec contexte) :")
                display(df_export.head(10))
                
                # Choisir le format d'export selon la taille
                if len(df_export) > self.max_excel_rows:
                    # Trop grand pour Excel → CSV + Parquet
                    print(f"\n   ⚠️  Fichier trop volumineux pour Excel ({len(df_export):,} lignes)")
                    print(f"   📦 Export en CSV et Parquet...")
                    
                    # Export CSV
                    filename_csv = f"{nom_variable}_{flag_name}_{self.timestamp}.csv"
                    filepath_csv = os.path.join(self.output_dir, filename_csv)
                    df_export.to_csv(filepath_csv, index=False, encoding='utf-8-sig')
                    print(f"   💾 CSV exporté : {filepath_csv}")
                    
                    # Export Parquet (plus compact)
                    filename_parquet = f"{nom_variable}_{flag_name}_{self.timestamp}.parquet"
                    filepath_parquet = os.path.join(self.output_dir, filename_parquet)
                    df_export.to_parquet(filepath_parquet, index=False, engine='pyarrow')
                    print(f"   💾 Parquet exporté : {filepath_parquet}")
                    
                    filename = filename_csv  # Pour le rapport
                    
                else:
                    # Taille OK pour Excel
                    filename = f"{nom_variable}_{flag_name}_{self.timestamp}.xlsx"
                    filepath = os.path.join(self.output_dir, filename)
                    df_export.to_excel(filepath, index=False, engine='openpyxl')
                    print(f"   💾 Excel exporté : {filepath}")
                
                print()
                
                # Stocker pour rapport global
                self.problemes_detectes.append({
                    'variable': nom_variable,
                    'flag': flag_name,
                    'description': config['description'],
                    'nb_observations': nb_observations_flagees,  # Observations réellement flaggées
                    'nb_matricules': nb_matricules,
                    'pct_observations': round(pct_obs_flagees, 2),
                    'pct_matricules': round(pct_mat, 2),
                    'fichier': filename
                })
        
        if not problemes_trouves:
            print("✅ Aucun problème détecté pour cette variable.\n")
    
    def generer_rapport_global(self):
        """
        Génère un rapport Excel consolidé de tous les problèmes détectés
        
        Le rapport contient :
        - Variable contrôlée
        - Flag déclenché
        - Description du problème
        - Nombre d'observations et matricules concernés
        - Pourcentages
        - Nom du fichier détaillé exporté
        """
        if not self.problemes_detectes:
            print("\n✅ AUCUN PROBLÈME DÉTECTÉ DANS TOUTE LA BASE !")
            return
        
        df_rapport = pd.DataFrame(self.problemes_detectes)
        
        filename = f"RAPPORT_GLOBAL_CONTROLES_{self.timestamp}.xlsx"
        filepath = os.path.join(self.output_dir, filename)
        
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            df_rapport.to_excel(writer, sheet_name='Synthèse', index=False)
        
        print("\n" + "="*80)
        print("📊 RAPPORT GLOBAL DES CONTRÔLES")
        print("="*80)
        display(df_rapport)
        print(f"\n💾 Rapport global exporté vers : {filepath}\n")

# SITUATION MATRIMONIALE

Cette section gère :
- le **recodage** de la situation matrimoniale,
- les **contrôles non temporels** (âge minimal au mariage),
- les **contrôles temporels** (variation de statut dans le temps).

⚠️ MODIFICATION :
- `FLAG_SITMAT_VARIATIONS` (multiples changements par an) a été **supprimé** car il générait trop de faux positifs.
- Flags conservés :
  - `FLAG_SITMAT_AGE` : marié(e) avant 18 ans → incohérence.
  - `FLAG_SITMAT_VARIATION` : changement de situation entre deux mois consécutifs.

---

1. `build_situation_matrimoniale(...)`

Recoder la situation matrimoniale à partir d’un mapping standard.

**Paramètres principaux :**
- `df` : DataFrame source.
- `source_col` : colonne d’origine (par défaut `"SITUATION MATRIMONIALE"`).
- `out_col` : colonne recodée (par défaut `"SITUATION_MATRIMONIALE_RECODE"`).
- `mapping` : dictionnaire de recodage (optionnel, sinon mapping par défaut).
- `return_report` : si `True`, renvoie aussi un petit rapport statistique.

**Mapping par défaut :**
- `"Célibataire"` → `"Célibataire"`
- `"Marié"`, `"Femme Mariée"`, `"Invalide Marié"` → `"Marié(e)"`
- `"Divorcé Séparé"` → `"Divorcé/Séparé"`
- `"Veuf / veuve"` → `"Veuf/Veuve"`
- `"Cas Particulier"` → `"Autres"`

**Sorties :**
- `df` avec une nouvelle colonne `out_col` recodée.
- `report` (si `return_report=True`) contenant :
  - un tableau `Effectif` / `Pourcentage (%)`,
  - les `value_counts` bruts et en pourcentage.

---

2. `sitmat_coherence_non_temporelle(...)`

Vérifier la **cohérence non temporelle** de la situation matrimoniale, sur une ligne prise isolément.

**Détection des colonnes :**
- Situation matrimoniale :
  - cherche d’abord parmi :
    - `"SITUATION MATRIMONIALE"`,
    - `"SITUATION_MATRIMONIALE"`,
    - `"SITUATION_MATRIMONIALE_RECODE"`,
  - sinon crée une colonne temporaire vide.
- Âge (`age_col`) :
  - priorise `"AGE_IMPUTE"`, sinon `"AGE"`, sinon colonne temporaire vide.
- Nombre d’enfants (`enfants_col`) :
  - cherche `"NOMBRE ENFANT"`, `"NB_ENFANTS"`, `"NBRE_ENFTS_IMPUTE"`, sinon colonne temporaire vide.

Toutes ces colonnes sont converties en numérique (`to_numeric`).

Normalisation de la situation matrimoniale
- `df['_SITMAT_NORM']` : texte normalisé via `_norm_txt` (minuscules, sans accents, etc.).
- `df['_SITMAT_RECODE']` : recodage dans quelques catégories standardisées :

  - Célibataire : `"celibataire"`, `"celib"` → `"celibataire"`
  - Marié(e) : `"marie"`, `"mariee"`, `"femme mariee"`, `"marie(e)"`, `"invalide marie"` → `"marie(e)"`
  - Union libre : `"union libre"`, `"concubinage"` → `"union libre"`
  - Divorcé / séparé : `"divorce"`, `"divorce separe"`, `"divorce/separe"` → `"divorce/separe"`
  - Veuf / veuve : `"veuf"`, `"veuve"`, `"veuf/veuve"` → `"veuf/veuve"`
  - Cas particuliers / autres : `"cas particulier"`, `"autre"`, `"autres"` → `"autres"`
  - Valeurs non reconnues → `"autres"` (via `fillna('autres')`).

Flag non temporel conservé

- `FLAG_SITMAT_AGE`  
  - Défini par :
    - situation recodée = `"marie(e)"`  
    - et `age_col < 18`.  
  - Interprétation : **mariage avant 18 ans** → incohérence potentielle.  
  - Type : entier `Int8` (0 / 1).

Flag non temporel désactivé

- `FLAG_SITMAT_ENFANTS` (commenté dans le code)  
  - Logique : `"celibataire"` avec nombre d’enfants > 0.  
  - Motif de désactivation : situation possible en pratique (célibataire ayant des enfants), donc trop de faux positifs.

**Sortie :**
- DataFrame copié avec :
  - `_SITMAT_NORM`
  - `_SITMAT_RECODE`
  - `FLAG_SITMAT_AGE`
  - (éventuelles colonnes temporaires si créées).

---

3. `sitmat_coherence_temporelle(df)`

Vérifier la **cohérence temporelle** de la situation matrimoniale, c’est-à-dire la façon dont elle évolue entre deux mois consécutifs.

Pré-requis
- `DATE_REF` doit exister :
  - si absente, elle est construite via `_ensure_date_ref(df)`.
- `_SITMAT_RECODE` doit être déjà présent :
  - donc il faut appeler `sitmat_coherence_non_temporelle` avant.

La fonction utilise `_need_keys(df)` pour vérifier la présence de :
- `MATRICULE`
- `DATE_REF`

Algorithme
1. Tri des données par `MATRICULE`, puis par `DATE_REF`.
2. Pour chaque `MATRICULE` :
   - création de `'_SITMAT_PREV'` = valeur `_SITMAT_RECODE` du mois précédent (via `groupby().shift(1)`).
3. Création du flag :

   - `FLAG_SITMAT_VARIATION` =
     - 1 si :
       - `_SITMAT_RECODE` ≠ `_SITMAT_PREV`  
       - `_SITMAT_RECODE` non nul  
       - `_SITMAT_PREV` non nul  
     - 0 sinon.

   → On détecte ainsi les **changements de statut matrimonial d’un mois à l’autre**, en ignorant les trous (NaN).

4. Suppression de la colonne temporaire `'_SITMAT_PREV'`.

Modifications importantes
- `FLAG_SITMAT_VARIATIONS` (ancien flag comptant les multiples changements de statut dans l’année) a été **supprimé** :
  - il produisait trop de faux positifs,
  - il est volontairement retiré de cette version.

**Sortie :**
- DataFrame trié, avec en plus :
  - `FLAG_SITMAT_VARIATION` indiquant les changements de statut d’un mois sur l’autre.

---

Résumé des flags utilisés pour la situation matrimoniale

- `FLAG_SITMAT_AGE`  
  - Non temporel.  
  - Marié(e) avec un âge strictement inférieur à 18 ans.  
  - Objectif : détecter les **mariages improbables** (erreurs d’âge ou de statut).

- `FLAG_SITMAT_VARIATION`  
  - Temporel.  
  - Changement de situation matrimoniale entre deux mois consécutifs pour un même matricule.  
  - Objectif : marquer les **ruptures de trajectoire** (mariage, divorce, veuvage, etc.), utiles en diagnostic.

- `FLAG_SITMAT_VARIATIONS`  
  - Ancien flag détectant les **multiples changements la même année**.  
  - Supprimé dans cette version car trop de faux positifs dans les données réelles.


In [4]:

# Cette partie contient le traitement de la situation matrimoniale.
# ⚠️ MODIFICATION : Le FLAG_SITMAT_VARIATIONS a été SUPPRIMÉ
# (trop de faux positifs)
#
# Flags conservés :
# - FLAG_SITMAT_AGE : Marié(e) avant 18 ans
# - FLAG_SITMAT_VARIATION : Changement entre deux mois
#

def build_situation_matrimoniale(df, source_col="SITUATION MATRIMONIALE", 
                                 out_col="SITUATION_MATRIMONIALE_RECODE",
                                 mapping=None, return_report=True):
    """
    Recoder la situation matrimoniale selon un mapping standard
    
    Args:
        df: DataFrame source
        source_col: Nom de la colonne source
        out_col: Nom de la colonne créée
        mapping: Dictionnaire de correspondance (optionnel)
        return_report: Retourner un rapport statistique
    
    Returns:
        df: DataFrame avec colonne recodée
        report: Rapport statistique (si return_report=True)
    
    Mapping par défaut:
        "Célibataire" → "Célibataire"
        "Marié", "Femme Mariée", "Invalide Marié" → "Marié(e)"
        "Divorcé Séparé" → "Divorcé/Séparé"
        "Veuf / veuve" → "Veuf/Veuve"
        "Cas Particulier" → "Autres"
    """
    if mapping is None:
        mapping = {
            "Célibataire": "Célibataire",
            "Cas Particulier": "Autres",
            "Divorcé Séparé": "Divorcé/Séparé",
            "Femme Mariée": "Marié(e)",
            "Invalide Marié": "Marié(e)",
            "Marié": "Marié(e)",
            "Veuf / veuve": "Veuf/Veuve"
        }

    out = df.copy()
    if source_col not in out.columns:
        raise KeyError(f"Colonne '{source_col}' absente du DataFrame.")

    out[out_col] = out[source_col].replace(mapping)

    if not return_report:
        return out

    eff = out[out_col].value_counts(dropna=False).sort_index()
    pct = out[out_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    report = {
        "table": pd.DataFrame({"Effectif": eff, "Pourcentage (%)": pct.round(2)}),
        "value_counts": eff,
        "value_counts_pct": pct.round(2)
    }

    return out, report


def sitmat_coherence_non_temporelle(df, age_col=None, enfants_col=None):
    """
    Vérifier la cohérence NON temporelle de la situation matrimoniale
    
    Contrôles effectués:
    1. FLAG_SITMAT_AGE : Marié(e) avant 18 ans
    2. FLAG_SITMAT_ENFANTS : Célibataire avec enfants (désactivé car situation possible)
    
    Args:
        df: DataFrame à contrôler
        age_col: Colonne d'âge (détection auto si None)
        enfants_col: Colonne nombre d'enfants (détection auto si None)
    
    Returns:
        DataFrame avec flags ajoutés et colonnes intermédiaires (_SITMAT_NORM, _SITMAT_RECODE)
    """
    df = df.copy()
    
    # Détecter la colonne de situation matrimoniale
    src = next((c for c in ['SITUATION MATRIMONIALE', 'SITUATION_MATRIMONIALE', 'SITUATION_MATRIMONIALE_RECODE'] if c in df.columns), None)
    if src is None:
        df['SITMAT_TMP'] = np.nan
        src = 'SITMAT_TMP'

    # Détecter la colonne d'âge
    if age_col is None:
        age_col = next((c for c in ['AGE_IMPUTE', 'AGE'] if c in df.columns), None)
        if age_col is None:
            df['AGE_TMP'] = np.nan
            age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    # Détecter la colonne nombre d'enfants
    if enfants_col is None:
        enfants_col = next((c for c in ['NOMBRE ENFANT', 'NB_ENFANTS', 'NBRE_ENFTS_IMPUTE'] if c in df.columns), None)
        if enfants_col is None:
            df['NB_ENFANTS_TMP'] = np.nan
            enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    # Normaliser la situation matrimoniale
    df['_SITMAT_NORM'] = df[src].map(_norm_txt)
    
    # Mapping vers catégories standardisées
    map_sit = {
        'celibataire': 'celibataire', 'celib': 'celibataire',
        'marie': 'marie(e)', 'mariee': 'marie(e)', 'femme mariee': 'marie(e)',
        'marie(e)': 'marie(e)',
        'invalide marie': 'marie(e)',
        'union libre': 'union libre', 'concubinage': 'union libre',
        'divorce': 'divorce/separe', 'divorce separe': 'divorce/separe',
        'divorce/separe': 'divorce/separe',
        'veuf': 'veuf/veuve', 'veuve': 'veuf/veuve', 'veuf/veuve': 'veuf/veuve',
        'cas particulier': 'autres', 'autre': 'autres', 'autres': 'autres'
    }
    df['_SITMAT_RECODE'] = df['_SITMAT_NORM'].replace(map_sit).fillna('autres')

    # FLAG 1 : Marié(e) avant 18 ans → incohérence
    df['FLAG_SITMAT_AGE'] = ((df['_SITMAT_RECODE'] == 'marie(e)') & (df[age_col] < 18)).astype('Int8')
    
    # FLAG 2 : Célibataire avec enfants → commenté car situation possible
    # df['FLAG_SITMAT_ENFANTS'] = ((df['_SITMAT_RECODE'] == 'celibataire') & (df[enfants_col] > 0)).astype('Int8')

    return df


def sitmat_coherence_temporelle(df):
    """
  
    Contrôle effectué:
    - FLAG_SITMAT_VARIATION : Changement entre deux mois consécutifs
    
    ⚠️ MODIFICATION : Le flag FLAG_SITMAT_VARIATIONS (multiples changements par an) 
    a été SUPPRIMÉ car il génère trop de faux positifs.
    
    Args:
        df: DataFrame à contrôler (doit avoir DATE_REF et _SITMAT_RECODE)
    
    Returns:
        DataFrame avec flag FLAG_SITMAT_VARIATION ajouté
    
    Algorithme:
        1. Trier par MATRICULE et DATE_REF
        2. Calculer la situation du mois précédent avec shift(1)
        3. Flaguer si changement entre deux valeurs non nulles
    """
    # Vérifier les prérequis
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    
    if '_SITMAT_RECODE' not in df.columns:
        raise ValueError("Appelle sitmat_coherence_non_temporelle d'abord.")

    print("   → Analyse des variations temporelles...")
    
    # Version optimisée : utiliser transform au lieu de apply
    df = df.sort_values(['MATRICULE', 'DATE_REF']).copy()
    
    # Calcul de la situation du mois précédent par matricule
    df['_SITMAT_PREV'] = df.groupby('MATRICULE')['_SITMAT_RECODE'].shift(1)
    
    # FLAG : Changement entre deux mois consécutifs
    df['FLAG_SITMAT_VARIATION'] = (
        (df['_SITMAT_RECODE'] != df['_SITMAT_PREV']) &  # Valeurs différentes
        df['_SITMAT_RECODE'].notna() &                  # Valeur actuelle non nulle
        df['_SITMAT_PREV'].notna()                      # Valeur précédente non nulle
    ).astype('Int8')
    
    # ✅ Code du FLAG_SITMAT_VARIATIONS SUPPRIMÉ (trop de faux positifs)
    
    # Nettoyage des colonnes temporaires
    df = df.drop(columns=['_SITMAT_PREV'], errors='ignore')
    
    return df

# NOMBRE D'ENFANTS

Cette section gère :

- la **construction** de la variable imputée `NBRE_ENFTS_IMPUTE`,
- les **contrôles non temporels** (cohérence avec l’âge, fertilité),
- les **contrôles temporels** (nombre d’enfants qui diminue dans le temps).

---

1. `build_nombre_enfants(...)`

Construit une variable propre de nombre d’enfants avec imputation simple.

**Paramètres principaux :**
- `df` : DataFrame source.
- `source_col` : colonne d’origine du nombre d’enfants (par défaut `"NOMBRE ENFANT"`).
- `out_col` : nom de la variable créée (par défaut `"NBRE_ENFTS_IMPUTE"`).
- `fill_value` : valeur utilisée pour imputer les manquants (par défaut `0`).
- `return_report` : si `True`, renvoie aussi un rapport statistique.

**Étapes de traitement :**
1. Vérification de l’existence de `source_col` :
   - si absente → `KeyError`.
2. Conversion vers numérique :
   - `pd.to_numeric(..., errors="coerce")` → valeurs non numériques transformées en `NaN`.
   - `fillna(fill_value)` pour imputer les manquants (par défaut 0).
3. Typage :
   - essai de conversion en entier `Int64` (arrondi préalable),
   - en cas d’erreur, retombe sur `Float64`.

**Sorties :**
- `df` avec une nouvelle colonne `out_col` (nombre d’enfants imputé et typé).
- `report` (si `return_report=True`) avec :
  - un tableau `Effectif` / `Pourcentage (%)`,
  - `value_counts` et `value_counts_pct`.

Variable centrale pour tous les contrôles liés aux enfants.

---

2. `enfants_coherence_non_temporelle(...)`

Vérifie la **cohérence non temporelle** entre le nombre d’enfants et l’âge de l’individu, observation par observation.

**Paramètres :**
- `df` : DataFrame à contrôler.
- `age_col` : colonne d’âge (détection automatique si `None`).
- `max_enfants` : (paramètre non utilisé dans le code actuel, laissé pour extension).

**Détection automatique des colonnes**

- Colonne du nombre d’enfants (`enfants_col`) :
  - recherche dans l’ordre :
    - `"NOMBRE ENFANT"`,
    - `"NB_ENFANTS"`,
    - `"NBRE_ENFTS_IMPUTE"`.
  - si aucune trouvée :
    - création d’une colonne temporaire `NB_ENFANTS_TMP` remplie de `NaN`.
- Colonne d’âge (`age_col`) :
  - recherche dans l’ordre :
    - `"AGE_IMPUTE"`,
    - `"AGE"`.
  - si aucune trouvée :
    - création d’une colonne temporaire `AGE_TMP` remplie de `NaN`.

Les deux colonnes sont converties en numérique (`to_numeric(..., errors='coerce')`).

**Flags non temporels**

- `FLAG_ENFANTS_AGE`  
  - Condition :
    - `age_col < 15`  
    - et `enfants_col > 0`.  
  - Interprétation :
    - **Enfant (moins de 15 ans) déclaré avec au moins un enfant** → incohérence forte.
  - Type : `Int8` (0 / 1).

- `FLAG_ENFANTS_FERT_IRR`  
  - Condition :
    - `enfants_col > (age_col - 12)`  
    - et `enfants_col` non manquant  
    - et `age_col` non manquant.  
  - Interprétation :
    - Nombre d’enfants **trop élevé au regard de l’âge** (implique une première naissance avant 12 ans ou moins),
    - Suspecte une **fertilité irréaliste** ou des erreurs de saisie.
  - Type : `Int8` (0 / 1).

**Colonne de traçabilité**

- `_ENF_COL` :
  - mémorise le nom effectif de la colonne utilisée pour le nombre d’enfants,
  - utilisée ensuite par `enfants_coherence_temporelle` pour éviter de repasser la logique de détection.

**Sortie :**
- DataFrame copié avec :
  - colonnes numériques d’âge et d’enfants,
  - `FLAG_ENFANTS_AGE`,
  - `FLAG_ENFANTS_FERT_IRR`,
  - `_ENF_COL`.

---

3. `enfants_coherence_temporelle(df)`

Vérifie la **cohérence temporelle** du nombre d’enfants dans la trajectoire d’un même individu (MATRICULE).

**Pré-requis :**
- `DATE_REF` doit exister :
  - si absente, création via `_ensure_date_ref(df)`.
- `MATRICULE` + `DATE_REF` vérifiés via `_need_keys(df)`.
- `_ENF_COL` doit exister :
  - donc il faut appeler `enfants_coherence_non_temporelle` avant.

**Algorithme (version optimisée)**

1. Copie du DataFrame et récupération du nom de la colonne nombre d’enfants :
   - `enfants_col = df['_ENF_COL'].iloc(0)`.
2. Tri des données par :
   - `MATRICULE`,
   - `DATE_REF`.
3. Pour chaque `MATRICULE` :
   - calcul de `'_ENF_PREV'` = nombre d’enfants à la période précédente (`groupby().shift(1)`).
4. Création du flag :

   - `FLAG_ENFANTS_DESCENTE` =
     - 1 si :
       - `enfants_col < _ENF_PREV`,
       - `enfants_col` non manquant,
       - `_ENF_PREV` non manquant.
     - 0 sinon.

   → Signifie que le **nombre d’enfants diminue dans le temps** pour un même individu, ce qui est en principe impossible.  
   → Fort indicateur d’erreur de saisie / d’update dans les fichiers sources.

5. Suppression de la colonne temporaire `'_ENF_PREV'`.

**Sortie :**
- DataFrame trié, avec en plus :
  - `FLAG_ENFANTS_DESCENTE`.

---

**Résumé des flags utilisés pour le nombre d’enfants**

- `FLAG_ENFANTS_AGE`  
  - Non temporel.  
  - Enfant (< 15 ans) déclaré avec au moins un enfant.  
  - Objectif : repérer des incohérences âge / statut parental.

- `FLAG_ENFANTS_FERT_IRR`  
  - Non temporel.  
  - Nombre d’enfants incompatible avec l’âge (trop élevé).  
  - Objectif : détecter des combinaisons âge / nombre d’enfants irréalistes.

- `FLAG_ENFANTS_DESCENTE`  
  - Temporel.  
  - Nombre d’enfants qui **diminue** d’une période à l’autre pour un même matricule.  
  - Objectif : repérer des problèmes de mise à jour ou de saisie dans les historiques.


In [5]:
# ============================================================
# PARTIE 4/8 : NOMBRE D'ENFANTS ET SEXE
# ============================================================
#
# Cette partie contient :
# - Le traitement du nombre d'enfants (imputation + contrôles)
# - Le traitement du sexe (nettoyage + imputation + contrôles)
#
# ============================================================

# ============================================================
# NOMBRE D'ENFANTS
# ============================================================

def build_nombre_enfants(df, source_col="NOMBRE ENFANT", out_col="NBRE_ENFTS_IMPUTE",
                        fill_value=0, return_report=True):
    """
    Créer la variable NBRE_ENFTS_IMPUTE avec imputation des valeurs manquantes
    
    Args:
        df: DataFrame source
        source_col: Colonne source
        out_col: Colonne créée
        fill_value: Valeur par défaut pour les manquants (défaut: 0)
        return_report: Retourner un rapport statistique
    
    Returns:
        df: DataFrame avec colonne imputée
        report: Rapport statistique (si return_report=True)
    
    Traitement:
        - Conversion en numérique
        - Remplissage des NaN par fill_value (0 par défaut)
        - Arrondissement à l'entier
        - Conversion en Int64 (ou Float64 si échec)
    """
    out = df.copy()
    if source_col not in out.columns:
        raise KeyError(f"Colonne '{source_col}' absente du DataFrame.")

    # Conversion numérique + imputation
    out[out_col] = pd.to_numeric(out[source_col], errors="coerce").fillna(fill_value)

    # Conversion en entier
    try:
        out[out_col] = out[out_col].round(0).astype("Int64")
    except Exception:
        out[out_col] = out[out_col].astype("Float64")

    if not return_report:
        return out

    # Générer le rapport
    eff = out[out_col].value_counts(dropna=False).sort_index()
    pct = out[out_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    report = {
        "table": pd.DataFrame({"Effectif": eff, "Pourcentage (%)": pct.round(2)}),
        "value_counts": eff,
        "value_counts_pct": pct.round(2)
    }

    return out, report


def enfants_coherence_non_temporelle(df, age_col=None, max_enfants=12):
    """
    Vérifier la cohérence NON temporelle du nombre d'enfants
    
    Contrôles effectués:
    1. FLAG_ENFANTS_AGE : Enfant(s) alors que l'individu a < 15 ans
    2. FLAG_ENFANTS_FERT_IRR : Nombre d'enfants > (âge - 12) → fertilité irréaliste
    
    Args:
        df: DataFrame à contrôler
        age_col: Colonne d'âge (détection auto si None)
        max_enfants: Seuil maximum d'enfants (non utilisé actuellement)
    
    Returns:
        DataFrame avec flags et colonne _ENF_COL (référence à la colonne utilisée)
    
    Exemples de problèmes détectés:
        - Personne de 14 ans avec 1 enfant → FLAG_ENFANTS_AGE = 1
        - Personne de 25 ans avec 15 enfants → FLAG_ENFANTS_FERT_IRR = 1
    """
    df = df.copy()
    
    # Détecter la colonne nombre d'enfants
    enfants_col = next((c for c in ['NOMBRE ENFANT', 'NB_ENFANTS', 'NBRE_ENFTS_IMPUTE'] if c in df.columns), None)
    if enfants_col is None:
        df['NB_ENFANTS_TMP'] = np.nan
        enfants_col = 'NB_ENFANTS_TMP'
    df[enfants_col] = pd.to_numeric(df[enfants_col], errors='coerce')

    # Détecter la colonne d'âge
    if age_col is None:
        age_col = next((c for c in ['AGE_IMPUTE', 'AGE'] if c in df.columns), None)
        if age_col is None:
            df['AGE_TMP'] = np.nan
            age_col = 'AGE_TMP'
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    # FLAG 1 : Enfants alors que trop jeune (< 15 ans)
    df['FLAG_ENFANTS_AGE'] = ((df[age_col] < 15) & (df[enfants_col] > 0)).astype('Int8')
    
    # FLAG 2 : Nombre d'enfants biologiquement irréaliste
    # Formule : nb_enfants > (age - 12)
    # Exemple : 25 ans → max 13 enfants théoriques (naissance à 12 ans + 1 par an)
    df['FLAG_ENFANTS_FERT_IRR'] = ((df[enfants_col] > (df[age_col] - 12)) & 
                                    df[enfants_col].notna() & df[age_col].notna()).astype('Int8')
    
    # Sauvegarder le nom de la colonne pour usage ultérieur
    df['_ENF_COL'] = enfants_col
    
    return df


def enfants_coherence_temporelle(df):
    """
    Vérifier la cohérence TEMPORELLE du nombre d'enfants - VERSION OPTIMISÉE
    
    Contrôle effectué:
    - FLAG_ENFANTS_DESCENTE : Nombre d'enfants qui diminue dans le temps
    
    Args:
        df: DataFrame à contrôler (doit avoir _ENF_COL défini)
    
    Returns:
        DataFrame avec flag ajouté
    
    Principe:
        Le nombre d'enfants ne peut QUE augmenter ou rester stable dans le temps.
        Une diminution indique une erreur de saisie ou de données.
    
    Exemple de problème:
        2020-01 : 3 enfants
        2020-02 : 2 enfants  ← FLAG levé car impossible biologiquement
    """
    # Vérifier les prérequis
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    
    if '_ENF_COL' not in df.columns:
        raise ValueError("Appelle enfants_coherence_non_temporelle d'abord.")
    
    print("   → Analyse des variations temporelles...")
    
    df = df.copy()
    enfants_col = df['_ENF_COL'].iloc[0]
    
    # Version optimisée avec groupby + shift
    df = df.sort_values(['MATRICULE', 'DATE_REF']).copy()
    df['_ENF_PREV'] = df.groupby('MATRICULE')[enfants_col].shift(1)
    
    # FLAG : Nombre d'enfants qui diminue
    df['FLAG_ENFANTS_DESCENTE'] = (
        (df[enfants_col] < df['_ENF_PREV']) &  # Diminution
        df[enfants_col].notna() &               # Valeur actuelle non nulle
        df['_ENF_PREV'].notna()                 # Valeur précédente non nulle
    ).astype('Int8')
    
    # Nettoyage
    df = df.drop(columns=['_ENF_PREV'], errors='ignore')
    
    return df



# SEXE

Cette section gère :
- le **nettoyage** du sexe à partir des observations disponibles,
- l’**imputation** des valeurs manquantes par mode mensuel,
- le **contrôle temporel** des variations de sexe dans le temps.

Elle produit notamment le flag :
- `FLAG_SEXE_VARIATION` : changement de sexe au cours du temps pour un même matricule.

---

1. `build_sexe_clean(...)`

Nettoyer et stabiliser le sexe au niveau du matricule à partir de l’historique.

**Paramètres principaux :**
- `df` : DataFrame source.
- `id_col` : identifiant de l’individu (par défaut `"MATRICULE"`).
- `sex_col` : colonne de sexe brute (par défaut `"SEXE"`).
- `collect_col` : colonne de date de collecte (par défaut `"DATE_COLLECTE"`).
- `new_col` : nom de la variable propre créée (par défaut `"SEXE_CLEAN"`).

**Étapes de traitement :**
1. Conversion de `DATE_COLLECTE` en `datetime` (`errors="coerce"`).
2. Création d’un index de ligne `_row_order` pour garder un ordre stable.
3. Construction d’un sous-ensemble `base` :
   - suppression des lignes où `collect_col` ou `sex_col` est manquant,
   - tri par `(MATRICULE, DATE_COLLECTE, _row_order)`.
4. Pour chaque `MATRICULE` :
   - on prend la **dernière** observation (`groupby().last()`),
   - on garde uniquement `(MATRICULE, SEXE)` et on renomme en `SEXE_CLEAN`.
5. Jointure sur le DataFrame d’origine :
   - suppression préalable de toute ancienne colonne `SEXE_CLEAN`,
   - merge `on=MATRICULE, how="left"`.

**Résultat :**
- `SEXE_CLEAN` est la **dernière valeur de sexe observée** pour chaque individu,
- prête à être utilisée comme variable de référence / d’entrée pour l’imputation.

---

2. `imputer_sexe(...)`

Imputer le sexe manquant en utilisant le **mode par mois / année de collecte**.

**Paramètres principaux :**
- `df` : DataFrame source.
- `sex_col` : colonne de sexe nettoyée (par défaut `"SEXE_CLEAN"`).
- `collect_col` : date de collecte (par défaut `"DATE_COLLECTE"`).
- `sex_valid_col` : nom de la colonne de sortie (par défaut `"SEXE_IMPUTE"`).
- `debug` : paramètre prévu pour des sorties de débogage (non utilisé dans cette version).

**Étapes de traitement :**
1. Conversion de `DATE_COLLECTE` en `datetime`.
2. Création de deux colonnes temporelles :
   - `ANNEE_COLLECTE` = année de la collecte,
   - `MOIS_COLLECTE` = mois de la collecte.
3. Initialisation de `SEXE_IMPUTE` :
   - copie directe de `sex_col` (ex. `SEXE_CLEAN`).
4. Calcul du **mode de sexe par (ANNÉE, MOIS)** :
   - on filtre sur les lignes où le sexe n’est pas manquant,
   - on groupe par `(ANNEE_COLLECTE, MOIS_COLLECTE)`,
   - pour chaque groupe, on prend le **mode** (valeur la plus fréquente),
   - si le mode est vide, on renvoie `NaN`.

5. Imputation ligne par ligne :
   - si `SEXE_IMPUTE` est **déjà renseigné** → on le garde,
   - si `SEXE_IMPUTE` est **NaN** :
     - on remplace par le mode du couple `(ANNEE_COLLECTE, MOIS_COLLECTE)` si disponible,
     - sinon, on garde la valeur actuelle (NaN).

**Résultat :**
- `SEXE_IMPUTE` :
  - variable de sexe consolidée,
  - sans (ou avec beaucoup moins de) valeurs manquantes,
  - cohérente avec la distribution mensuelle des données.

---

3. `sexe_coherence_temporelle(df)`

Vérifier la **cohérence temporelle** du sexe pour chaque individu, c’est-à-dire détecter les cas où le sexe change dans le temps.

**Pré-requis :**
- `DATE_REF` doit exister :
  - si absente, elle est créée via `_ensure_date_ref(df)`.
- `MATRICULE` + `DATE_REF` sont vérifiés par `_need_keys(df)`.
- Au moins une colonne de sexe parmi :
  - `"SEXE_IMPUTE"`,
  - `"SEXE_CLEAN"`,
  - `"SEXE"`.

**Étapes de traitement**

1. **Choix de la colonne de sexe** :
   - on prend la première disponible dans l’ordre :
     - `SEXE_IMPUTE`,
     - sinon `SEXE_CLEAN`,
     - sinon `SEXE`,
   - si aucune trouvée → `ValueError`.

2. **Normalisation du sexe** :
   - `df['_SEXE_NORM'] = _sex_norm(df[sex_col])`
   - `_sex_norm` recode tout en `{M, F, NaN}` :
     - `"MASCULIN"`, `"HOMME"`, `"H"` → `M`,
     - `"FEMININ"`, `"FEMME"` → `F`,
     - autres valeurs → `NaN`.

3. **Tri temporel** :
   - tri par `(MATRICULE, DATE_REF)` pour reconstituer la trajectoire.

4. **Comparaison avec la période précédente** :
   - création de `'_SEXE_PREV'` via :
     - `groupby('MATRICULE')['_SEXE_NORM'].shift(1)`,
   - `FLAG_SEXE_VARIATION` =
     - 1 si :
       - `_SEXE_NORM` ≠ `_SEXE_PREV`,
       - `_SEXE_NORM` non nul,
       - `_SEXE_PREV` non nul,
     - 0 sinon.

5. **Nettoyage** :
   - suppression de la colonne temporaire `'_SEXE_PREV'`.

**Résultat :**
- DataFrame enrichi avec :
  - `_SEXE_NORM` (sexe normalisé),
  - `FLAG_SEXE_VARIATION` (variation de sexe dans le temps).

---

**Résumé des variables et flags pour le sexe**

- `SEXE_CLEAN`  
  - Dernière valeur de sexe observée pour chaque matricule (stabilisation “dernière observation connue”).

- `SEXE_IMPUTE`  
  - Sexe consolidé après imputation des valeurs manquantes,
  - basé sur le mode par `(ANNEE_COLLECTE, MOIS_COLLECTE)`.

- `FLAG_SEXE_VARIATION`  
  - Indicateur temporel,
  - vaut 1 si le sexe change entre deux périodes consécutives (pour un même matricule),
  - signale des incohérences fortes (changement de sexe “administratif”) ou des erreurs de saisie.


In [6]:
def build_sexe_clean(df, id_col="MATRICULE", sex_col="SEXE", 
                     collect_col="DATE_COLLECTE", new_col="SEXE_CLEAN"):
    """
    Nettoyer le sexe en prenant la dernière valeur renseignée par matricule
    
    Args:
        df: DataFrame source
        id_col: Colonne d'identifiant
        sex_col: Colonne de sexe source
        collect_col: Colonne de date de collecte
        new_col: Colonne créée
    
    Returns:
        DataFrame avec colonne SEXE_CLEAN ajoutée
    
    Principe:
        Le sexe ne devrait pas changer. On prend la dernière valeur
        renseignée dans le temps pour chaque matricule.
    
    Algorithme:
        1. Trier par MATRICULE et DATE_COLLECTE (décroissant)
        2. Garder la dernière ligne de chaque matricule
        3. Merger cette valeur sur toutes les observations du matricule
    """
    df = df.copy()
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")
    df["_row_order"] = np.arange(len(df))

    # Trier et prendre la dernière valeur par matricule
    base = df.dropna(subset=[collect_col, sex_col]).sort_values([id_col, collect_col, "_row_order"])
    latest = base.groupby(id_col, as_index=False).last()[[id_col, sex_col]].rename(columns={sex_col: new_col})

    # Merger sur toutes les lignes
    df = df.drop(columns=[new_col], errors="ignore").merge(latest, on=id_col, how="left")
    return df.drop(columns=["_row_order"])


def imputer_sexe(df, sex_col="SEXE_CLEAN", collect_col="DATE_COLLECTE", 
                 sex_valid_col="SEXE_IMPUTE", debug=False):
    """
    Imputer le sexe par le mode du groupe (année, mois) ou par le mode global
    
    Args:
        df: DataFrame source
        sex_col: Colonne de sexe à imputer
        collect_col: Colonne de date
        sex_valid_col: Colonne créée avec valeurs imputées
        debug: Afficher des informations de débogage
    
    Returns:
        DataFrame avec colonne SEXE_IMPUTE ajoutée
    
    Stratégie d'imputation:
        1. Calculer le mode (valeur la plus fréquente) par (année, mois)
        2. Pour chaque valeur manquante, imputer avec le mode de son groupe
        3. Si le mode du groupe est NaN, ne pas imputer
    
    Exemple:
        2020-01 : M=80%, F=20% → Mode = M
        2020-01 : NaN → Imputé à M
    """
    df = df.copy()
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # Extraire année et mois
    df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    df["MOIS_COLLECTE"] = df[collect_col].dt.month
    df[sex_valid_col] = df[sex_col]

    # Calculer le mode par groupe (année, mois)
    mode_sexe_groupes = (
        df.dropna(subset=[sex_col])
          .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])[sex_col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    # Imputer les valeurs manquantes
    df[sex_valid_col] = df.apply(
        lambda row: mode_sexe_groupes.get(
            (row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), row[sex_valid_col]
        ) if pd.isna(row[sex_valid_col]) else row[sex_valid_col],
        axis=1
    )

    return df


def sexe_coherence_temporelle(df):
    """
    Vérifier la cohérence TEMPORELLE du sexe - VERSION OPTIMISÉE
    
    Contrôle effectué:
    - FLAG_SEXE_VARIATION : Sexe qui varie dans le temps
    
    Args:
        df: DataFrame à contrôler
    
    Returns:
        DataFrame avec flag ajouté
    
    Principe:
        Le sexe biologique ne change pas. Toute variation indique
        une erreur de saisie grave.
    
    Exemple de problème:
        2020-01 : M
        2020-02 : F  ← FLAG levé car variation impossible
    """
    # Vérifier les prérequis
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    
    print("   → Analyse des variations temporelles...")
    
    # Détecter la colonne de sexe
    sex_col = next((c for c in ['SEXE_IMPUTE', 'SEXE_CLEAN', 'SEXE'] if c in df.columns), None)
    if sex_col is None:
        raise ValueError("Aucune colonne de sexe trouvée.")
    
    df = df.copy()
    df['_SEXE_NORM'] = _sex_norm(df[sex_col])
    
    # Version optimisée
    df = df.sort_values(['MATRICULE', 'DATE_REF']).copy()
    df['_SEXE_PREV'] = df.groupby('MATRICULE')['_SEXE_NORM'].shift(1)
    
    # FLAG : Sexe qui change
    df['FLAG_SEXE_VARIATION'] = (
        (df['_SEXE_NORM'] != df['_SEXE_PREV']) &
        df['_SEXE_NORM'].notna() & 
        df['_SEXE_PREV'].notna()
    ).astype('Int8')
    
    # Nettoyage
    df = df.drop(columns=['_SEXE_PREV'], errors='ignore')
    
    return df

# ÂGE

Cette section gère :
- la **construction d’un âge précis** à partir de la date de naissance et de la date de collecte,
- la création des variables `AGE`, `AGE_VALIDE`, `AGE_IMPUTE`,
- une **imputation temporellement cohérente** de l’âge dans le temps pour chaque matricule,
- une **stabilisation** de l’âge imputé au niveau individuel,
- des **contrôles temporels** de cohérence de l’évolution de l’âge.

Elle introduit notamment les flags :
- `FLAG_AGE_DECROIT` : âge qui diminue dans le temps,
- `FLAG_AGE_TROP_RAPIDE` : âge qui augmente trop vite par rapport au temps écoulé.

---

1. `build_age_valide(...)`

Construit les variables d’âge et réalise une imputation **temporellement cohérente**.

**Paramètres principaux :**
- `df` : DataFrame source.
- `matricule_col` : identifiant de l’individu (par défaut `"MATRICULE"`).
- `birth_col` : colonne de date de naissance (par défaut `"DATE NAISSANCE"`).
- `collect_col` : colonne de date de collecte (par défaut `"DATE_COLLECTE"`).
- `sex_col` : colonne de sexe imputé (par défaut `"SEXE_IMPUTE"`) utilisée pour les groupes d’imputation.
- `age_min`, `age_max` : bornes de validité de l’âge (par défaut 18–70 ans).
- `do_impute_age` : si `True`, impute l’âge manquant.
- `stabilize_age` : si `True`, ajoute une version stabilisée de l’âge imputé.
- `stabilize_method` : méthode de stabilisation (`"last"`, `"mode"`, `"median"`).

1.1 Préparation des dates et variables de collecte

- Conversion de `DATE_COLLECTE` en datetime via `_to_datetime_mixed`.
- Création de :
  - `ANNEE_COLLECTE` = année de collecte (si absente),
  - `MOIS_COLLECTE` = mois de collecte (si absent).

Ces variables servent à structurer l’imputation par groupe temporel.

1.2 Nettoyage de la date de naissance

- `DATE_NAISSANCE_CLEAN` :
  - obtenue via `_clean_dates_generic(birth_col)` :
    - gère formats mixtes, dates nulles, numéros Excel, etc.
- Pour chaque `MATRICULE` :
  - on garde la **date de naissance la plus récente observée** dans les données :
    - tri par `(MATRICULE, DATE_COLLECTE)` en ordre décroissant de date,
    - `drop_duplicates` sur `MATRICULE` en gardant la première,
    - renommage en `DATE_NAISSANCE_CORRIGEE`.

Résultat : `DATE_NAISSANCE_CORRIGEE` est une date de naissance stabilisée pour chaque individu.

1.3 Calcul de l’âge précis (`AGE`)

On calcule un âge exact en années, en tenant compte du mois et du jour d’anniversaire.

- Pour les lignes où `DATE_NAISSANCE_CORRIGEE` et `DATE_COLLECTE` sont non nulles :
  - `ydiff` = année collecte – année naissance,
  - `before_bday` = indicateur “l’anniversaire de l’année n’est pas encore passé” :
    - `True` si le mois/jour de collecte est avant le mois/jour de naissance,
    - `False` sinon.
  - `AGE` = `ydiff - before_bday` (converti en float).

**Résultat :**
- `df["AGE"]` : âge réel (en années entières) au moment de la collecte.

1.4 Définition de l’âge valide (`AGE_VALIDE`)

- `AGE_VALIDE` = `AGE` si `AGE` ∈ `[age_min ; age_max]`,
- sinon → `NaN`.

Cela permet d’isoler un noyau d’âges plausibles servant de base à l’imputation.

---

1.5 Imputation temporellement cohérente (`AGE_IMPUTE`)

Si `do_impute_age=True` :

1. **Tri des données** :
   - tri par `(MATRICULE, DATE_COLLECTE)`.

2. **Étape 1 – Imputation initiale par médiane de groupe** :

   - Construction de `group_keys` :
     - `["ANNEE_COLLECTE", "MOIS_COLLECTE"]`,
     - + `SEXE_IMPUTE` si la colonne `sex_col` existe.
   - `med` = médiane de `AGE_VALIDE` par groupe `(année, mois, sexe?)` via `groupby(...).transform('median')`.
   - `global_med` = médiane globale de `AGE_VALIDE`.
   - `AGE_IMPUTE_BASE` :
     - si `AGE_VALIDE` non nul → on garde `AGE_VALIDE`,
     - sinon → on remplace par `med`, avec repli sur `global_med`.

   → On obtient une première approximation d’âge imputé, par groupes homogènes.

3. **Étape 2 – Propagation temporelle cohérente par matricule**

   La fonction interne `_impute_coherent(g)` est appliquée matricule par matricule :

   - Le groupe `g` est trié par `DATE_COLLECTE`.
   - Cas 1 : le matricule possède au moins un `AGE_VALIDE` non nul :
     - on identifie la **première observation** avec `AGE_VALIDE` non manquant :
       - `base_age` = valeur d’âge à cette date de référence,
       - `base_date` = date de collecte correspondante.
     - pour chaque observation du groupe :
       - `time_diff_years` = (DATE_COLLECTE – base_date) / 365,25,
       - `AGE_IMPUTE` = `base_age + time_diff_years`, arrondi à l’entier.
   - Cas 2 : aucun `AGE_VALIDE` non manquant pour le matricule :
     - on utilise `AGE_IMPUTE_BASE` comme base,
     - `base_age_imputed` = première valeur de `AGE_IMPUTE_BASE`,
     - `first_date` = première `DATE_COLLECTE`,
     - progression temporelle :
       - `time_diff` = (DATE_COLLECTE – first_date) / 365,25,
       - `AGE_IMPUTE` = `base_age_imputed + time_diff`, arrondi.

   → Objectif : garantir que l’âge progresse de façon **monotone et cohérente** avec le temps, même en présence de trous.

4. **Post-traitement :**
   - `AGE_IMPUTE` converti en type entier `Int64`,
   - suppression de la colonne `AGE_IMPUTE_BASE`.

**Résultat :**
- `AGE_IMPUTE` : âge imputé et cohérent avec l’axe temporel (ici, approximativement +1 an par an).

---

1.6 Stabilisation de l’âge imputé (`AGE_IMPUTE_STABLE`)

Si `stabilize_age=True` :

- On regroupe par `MATRICULE` et on applique une méthode de stabilisation sur `AGE_IMPUTE` :

  - `stabilize_method="last"` :
    - `AGE_IMPUTE_STABLE` = **dernier** âge imputé observé pour le matricule.
  - `stabilize_method="mode"` :
    - `AGE_IMPUTE_STABLE` = **mode** (âge le plus fréquent) via `_mode_safe`.
  - `stabilize_method="median"` :
    - `AGE_IMPUTE_STABLE` = **médiane** des âges imputés du matricule.

- Le résultat est fusionné dans `df` via `merge(...)`.

**Utilité :**
- disposer d’un âge unique par individu pour les analyses de stock, ou comme référence dans d’autres contrôles.

---

2. `age_coherence_temporelle(...)`

Vérifie la **cohérence temporelle de l’âge**, c’est-à-dire la façon dont l’âge évolue au fil des `DATE_REF` pour un même individu.

**Paramètres :**
- `df` : DataFrame à contrôler.
- `max_drift` : tolérance en années au-delà de la dérive temporelle attendue (par défaut 1.0 an).

**Notes sur la tolérance :**
- tient compte :
  - des décalages liés aux dates d’anniversaire,
  - des approximations / arrondis,
  - des effets des imputations.

2.1 Pré-requis

- `DATE_REF` doit exister :
  - sinon, création via `_ensure_date_ref(df)`.
- `MATRICULE` + `DATE_REF` vérifiés via `_need_keys(df)`.

- Colonne d’âge utilisée (`age_col`) :
  - première trouvée parmi :
    - `"AGE_IMPUTE"`,
    - `"AGE"`.
  - si aucune trouvée → `ValueError`.

2.2 Calcul des deltas temporels

- Tri du DataFrame par `(MATRICULE, DATE_REF)`.
- Pour chaque `MATRICULE` :
  - `'_DATE_PREV'` = date de référence précédente (`shift(1)`),
  - `'_YEAR_DELTA'` = (DATE_REF – `_DATE_PREV`) / 365,25,
  - les valeurs initiales (sans date précédente) reçoivent `0`.


2.3 Calcul des flags

- `'_AGE_PREV'` = âge à la période précédente (via `groupby().shift(1)`).

**FLAG 1 : `FLAG_AGE_DECROIT`**

- Condition :
  - `age_col < _AGE_PREV`,
  - `age_col` non nul,
  - `_AGE_PREV` non nul.
- Interprétation :
  - **L’âge diminue dans le temps** → incohérence forte (erreur de saisie ou d’imputation).

**FLAG 2 : `FLAG_AGE_TROP_RAPIDE`**

- Condition :
  - `(age_col - _AGE_PREV) > (_YEAR_DELTA + max_drift)`,
  - `age_col` non nul,
  - `_AGE_PREV` non nul.
- Interprétation :
  - l’augmentation de l’âge est **plus rapide que le temps écoulé + la tolérance**,  
  - par exemple, +3 ans sur une période de 1 an réel,
  - signale des **erreurs d’âge** ou de date.

2.4 Nettoyage

- Suppression de `'_AGE_PREV'` (colonnes temporaires de calcul).

**Résultat :**
- DataFrame enrichi avec :
  - `FLAG_AGE_DECROIT`,
  - `FLAG_AGE_TROP_RAPIDE`,
  - `'_DATE_PREV'` et `'_YEAR_DELTA'` peuvent être conservées ou ignorées selon usage (ici `_DATE_PREV` est conservée, `_AGE_PREV` est supprimée).



In [7]:
def build_age_valide(df, matricule_col="MATRICULE", birth_col="DATE NAISSANCE",
                     collect_col="DATE_COLLECTE", sex_col="SEXE_IMPUTE",
                     age_min=18, age_max=70, do_impute_age=True,
                     stabilize_age=True, stabilize_method="last"):
    """
    Construire AGE, AGE_VALIDE, AGE_IMPUTE avec imputation temporellement cohérente
    
    Cette fonction est le cœur du traitement de l'âge. Elle permet de :
    1. Calculer l'âge précis depuis la date de naissance
    2. Valider l'âge (dans l'intervalle [age_min, age_max])
    3. Imputer les âges manquants de manière cohérente dans le temps
    4. Stabiliser l'âge par matricule (optionnel)
    
    Args:
        df: DataFrame source
        matricule_col: Colonne d'identifiant
        birth_col: Colonne de date de naissance
        collect_col: Colonne de date de collecte
        sex_col: Colonne de sexe (pour imputation)
        age_min: Âge minimum valide (défaut: 18)
        age_max: Âge maximum valide (défaut: 70)
        do_impute_age: Effectuer l'imputation (défaut: True)
        stabilize_age: Stabiliser par matricule (défaut: True)
        stabilize_method: Méthode de stabilisation ('last', 'mode', 'median')
    
    Returns:
        DataFrame avec colonnes ajoutées:
        - AGE : Âge calculé depuis date de naissance
        - AGE_VALIDE : Âge validé (entre age_min et age_max)
        - AGE_IMPUTE : Âge imputé et cohérent temporellement
        - AGE_IMPUTE_STABLE : Âge stabilisé par matricule (optionnel)
    
    Algorithme d'imputation:
        1. Calculer médiane par (année, mois, sexe)
        2. Pour chaque matricule:
           a. Si âge valide existe → propager dans le temps
           b. Sinon → utiliser médiane + progression temporelle
        3. Garantir que l'âge augmente correctement dans le temps
    """
    df = df.copy()
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    
    # Créer année et mois de collecte si manquants
    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # Nettoyage et correction de la date de naissance
    df["DATE_NAISSANCE_CLEAN"] = _clean_dates_generic(df[birth_col])
    
    # Garder la dernière date de naissance renseignée par matricule
    tmp = (df.dropna(subset=["DATE_NAISSANCE_CLEAN"])
           .sort_values([matricule_col, collect_col], ascending=[True, False])
           .drop_duplicates(subset=[matricule_col], keep="first")
           [[matricule_col, "DATE_NAISSANCE_CLEAN"]]
           .rename(columns={"DATE_NAISSANCE_CLEAN": "DATE_NAISSANCE_CORRIGEE"}))
    
    df = df.drop(columns=["DATE_NAISSANCE_CORRIGEE"], errors="ignore").merge(tmp, on=matricule_col, how="left", validate="many_to_one")
    df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(df["DATE_NAISSANCE_CORRIGEE"], errors="coerce")

    # Calcul de l'âge précis (années révolues)
    birth = df["DATE_NAISSANCE_CORRIGEE"]
    ref = df[collect_col]
    age = pd.Series(pd.NA, index=df.index, dtype="Float64")
    mask = birth.notna() & ref.notna()
    
    if mask.any():
        # Différence d'années
        ydiff = ref[mask].dt.year - birth[mask].dt.year
        # Ajuster si anniversaire pas encore passé
        before_bday = ((ref[mask].dt.month < birth[mask].dt.month) | 
                      ((ref[mask].dt.month == birth[mask].dt.month) & (ref[mask].dt.day < birth[mask].dt.day)))
        age.loc[mask] = (ydiff - before_bday.astype(int)).astype("Float64")
    
    df["AGE"] = age
    df["AGE_VALIDE"] = df["AGE"].where(df["AGE"].between(age_min, age_max))

    # IMPUTATION TEMPORELLEMENT COHÉRENTE
    if do_impute_age:
        print("   → Imputation temporellement cohérente de l'âge...")
        
        # Trier par matricule et date
        df = df.sort_values([matricule_col, collect_col]).copy()
        
        # Étape 1 : Calculer médiane par groupe
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE"]
        if sex_col in df.columns:
            group_keys.append(sex_col)
        
        med = df.groupby(group_keys)["AGE_VALIDE"].transform("median")
        global_med = df["AGE_VALIDE"].median()
        
        # Créer AGE_IMPUTE_BASE (première imputation)
        df["AGE_IMPUTE_BASE"] = df["AGE_VALIDE"].where(df["AGE_VALIDE"].notna(), med.fillna(global_med))
        
        # Étape 2 : Propager l'âge de manière cohérente dans le temps par matricule
        def _impute_coherent(g):
            """
            Pour chaque matricule, propager l'âge de manière temporellement cohérente
            
            Principe:
            1. Si on a un âge valide à une date donnée
            2. On calcule l'âge pour toutes les autres dates en ajoutant le temps écoulé
            3. Ainsi l'âge augmente correctement dans le temps
            """
            g = g.sort_values(collect_col).copy()
            
            # Si on a au moins un âge valide, on l'utilise comme base
            if g["AGE_VALIDE"].notna().any():
                # Trouver la première observation avec âge valide
                first_valid_idx = g["AGE_VALIDE"].first_valid_index()
                if first_valid_idx is not None:
                    base_age = g.loc[first_valid_idx, "AGE_VALIDE"]
                    base_date = g.loc[first_valid_idx, collect_col]
                    
                    # Calculer l'âge pour toutes les observations en fonction du temps écoulé
                    time_diff_years = (g[collect_col] - base_date).dt.days / 365.25
                    g["AGE_IMPUTE"] = (base_age + time_diff_years).round(0)
                else:
                    # Pas d'âge valide, utiliser la médiane et progresser dans le temps
                    g["AGE_IMPUTE"] = g["AGE_IMPUTE_BASE"]
                    # Ajouter le temps écoulé depuis la première observation
                    first_date = g[collect_col].iloc[0]
                    time_diff = (g[collect_col] - first_date).dt.days / 365.25
                    g["AGE_IMPUTE"] = (g["AGE_IMPUTE"] + time_diff).round(0)
            else:
                # Aucun âge valide pour ce matricule, utiliser la médiane + progression temporelle
                base_age_imputed = g["AGE_IMPUTE_BASE"].iloc[0]
                first_date = g[collect_col].iloc[0]
                time_diff = (g[collect_col] - first_date).dt.days / 365.25
                g["AGE_IMPUTE"] = (base_age_imputed + time_diff).round(0)
            
            return g
        
        # Appliquer l'imputation cohérente par matricule
        tdf = df.groupby(matricule_col, group_keys=False).apply(_impute_coherent)
        
        # Restaurer l'index original si nécessaire
        if not tdf.index.equals(df.index):
            tdf = tdf.reset_index(drop=True)
            df.index = tdf.index
        
        df = tdf.copy()
        
        # Nettoyer et convertir
        df["AGE_IMPUTE"] = df["AGE_IMPUTE"].astype("Int64")
        df = df.drop(columns=["AGE_IMPUTE_BASE"], errors="ignore")

    # Stabilisation (optionnelle) : une seule valeur d'âge par matricule
    if stabilize_age:
        g = df.sort_values([matricule_col, collect_col]).groupby(matricule_col)["AGE_IMPUTE"]
        if stabilize_method == "last":
            stable = g.last()
        elif stabilize_method == "mode":
            stable = g.apply(_mode_safe)
        elif stabilize_method == "median":
            stable = g.median()
        else:
            raise ValueError("stabilize_method ∈ {'last','mode','median'}")
        
        df = df.merge(stable.rename("AGE_IMPUTE_STABLE"), on=matricule_col, how="left")

    return df


def age_coherence_temporelle(df, max_drift=1.0):
    """
    Vérifier la cohérence TEMPORELLE de l'âge - VERSION OPTIMISÉE
    
    Tolérance de 1 an pour tenir compte :
    - Des dates d'anniversaire (écart année civile vs âge réel)
    - Des imputations qui peuvent avoir un léger décalage
    - Des arrondis dans le calcul d'âge
    
    Contrôles effectués:
    1. FLAG_AGE_DECROIT : Âge qui diminue (toujours problématique)
    2. FLAG_AGE_TROP_RAPIDE : Âge qui augmente trop vite
    
    Args:
        df: DataFrame à contrôler
        max_drift: Tolérance en années (défaut: 1.0)
    
    Returns:
        DataFrame avec flags ajoutés
    
    Exemples:
        Temps écoulé = 1 an, Âge augmente de 2 ans → OK (tolérance 1 an)
        Temps écoulé = 1 an, Âge augmente de 3 ans → FLAG (dépassement tolérance)
    """
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    
    print("   → Analyse des variations temporelles...")
    
    age_col = next((c for c in ['AGE_IMPUTE', 'AGE'] if c in df.columns), None)
    if age_col is None:
        raise ValueError("Aucune colonne d'âge trouvée.")
    
    df = df.copy()
    df = df.sort_values(['MATRICULE', 'DATE_REF']).copy()
    
    # Calcul du delta temporel (en années)
    df['_DATE_PREV'] = df.groupby('MATRICULE')['DATE_REF'].shift(1)
    df['_YEAR_DELTA'] = ((df['DATE_REF'] - df['_DATE_PREV']).dt.days / 365.25).fillna(0)
    
    # Calcul de l'âge précédent
    df['_AGE_PREV'] = df.groupby('MATRICULE')[age_col].shift(1)
    
    # FLAG 1 : Âge qui diminue (toujours problématique)
    df['FLAG_AGE_DECROIT'] = (
        (df[age_col] < df['_AGE_PREV']) & 
        df[age_col].notna() & 
        df['_AGE_PREV'].notna()
    ).astype('Int8')
    
    # FLAG 2 : Âge qui augmente trop vite (> temps écoulé + tolérance)
    df['FLAG_AGE_TROP_RAPIDE'] = (
        ((df[age_col] - df['_AGE_PREV']) > (df['_YEAR_DELTA'] + max_drift)) &
        df[age_col].notna() & 
        df['_AGE_PREV'].notna()
    ).astype('Int8')
    
    df = df.drop(columns=['_AGE_PREV'], errors='ignore')
    
    return df

# ÂGE DE PRISE DE SERVICE

Cette section gère :
- la **construction d’un âge au service STABLE** (`AGE_AU_SERVICE`, `AGE_AU_SERVICE_VALIDE`) à partir de la date de prise de service et de l’âge,
- des **diagnostics détaillés** (méthode utilisée, âges négatifs, statistiques globales),
- les **contrôles non temporels** de l’âge de prise de service,
- les **contrôles temporels** de la stabilité de l’âge de prise de service dans le temps.

Elle introduit notamment les flags :
- `FLAG_AGE_PS_NEGATIF` : âge au service négatif,
- `FLAG_AGE_PS_GE_AGE` : âge au service supérieur ou égal à l’âge actuel,
- `FLAG_AGE_PS_VARIATION` : variation de l’âge au service au cours du temps.

---

1. `build_age_service_stable(...)`

Construit un **âge au service STABLE** (une seule valeur par matricule), de manière robuste, en gérant explicitement les cas d’âges négatifs.

**Paramètres principaux :**
- `df` : DataFrame source.
- `matricule_col` : identifiant de l’individu (par défaut `"MATRICULE"`).
- `start_col_corrected` : colonne de prise de service corrigée (par défaut `"PRISE_DE_SERVICE_CORRIGEE"`).
- `collect_col` : colonne de date de collecte (par défaut `"DATE_COLLECTE"`).
- `age_src_priority` : ordre de préférence des colonnes d’âge (`("AGE_IMPUTE", "AGE")` par défaut).
- `age_min`, `age_max` : bornes de validité de l’âge au service (par défaut 14–60 ans).

1.1 Préparation des dates de collecte et de prise de service

- Conversion de `DATE_COLLECTE` en datetime via `_to_datetime_mixed`.
- Si `start_col_corrected` n’existe pas :
  - création de `PRISE_DE_SERVICE_CLEAN` via `_clean_dates_generic("PRISE DE SERVICE")`,
  - tri par `(MATRICULE, DATE_COLLECTE)` en ordre décroissant,
  - pour chaque `MATRICULE`, on prend la **dernière valeur observée** de prise de service,
  - renommage en `PRISE_DE_SERVICE_CORRIGEE` et jointure sur le DataFrame.

- `PRISE_DE_SERVICE_CORRIGEE` est ensuite reconvertie en datetime via `_to_datetime_mixed`.

1.2 Sélection de la source d’âge

- Parcours de `age_src_priority` (`"AGE_IMPUTE"`, puis `"AGE"`):
  - première colonne existante choisie comme `age_src`,
  - message informatif :  
    `→ Utilisation de 'AGE_IMPUTE' pour le calcul`, par exemple.
- Si aucune colonne d’âge trouvée → `ValueError`.

1.3 Calcul robuste de l’âge au service par matricule

Les données sont triées par `(MATRICULE, DATE_COLLECTE)`, puis on applique `_calc_age_service_robust` **par groupe** (un groupe = un matricule).

Logique de `_calc_age_service_robust(g)`

1. **Filtrage des observations valides** (`valides`) :
   - `age_src` non nul,
   - `PRISE_DE_SERVICE_CORRIGEE` non nulle,
   - `DATE_COLLECTE` non nulle.
   - Si `valides` est vide :
     - `AGE_AU_SERVICE` = `NaN`,
     - `AGE_AU_SERVICE_VALIDE` = `NaN`,
     - `AGE_SERVICE_METHODE` = `"AUCUNE_DONNEE"`.

2. **Méthode 1 – Première observation (standard)**  
   - `first_obs` = première ligne de `valides` (dans l’ordre trié).
   - `age_first` = `age_src` à cette date.
   - `date_first` = `DATE_COLLECTE` de cette observation.
   - `prise_service` = `PRISE_DE_SERVICE_CORRIGEE` (prise de service utilisée pour toutes les méthodes).
   - `anciennete_first` = (date_first – prise_service) / 365,25.
   - `age_service_method1` = `age_first - anciennete_first`.

3. **Méthode 2 – Observation médiane**  
   - `mid_obs` = observation au milieu de `valides`.
   - `age_mid` = `age_src` de `mid_obs`.
   - `date_mid` = `DATE_COLLECTE` de `mid_obs`.
   - `anciennete_mid` = (date_mid – prise_service) / 365,25.
   - `age_service_method2` = `age_mid - anciennete_mid`.

4. **Méthode 3 – Dernière observation**  
   - `last_obs` = dernière observation de `valides`.
   - `age_last` = `age_src` de `last_obs`.
   - `date_last` = `DATE_COLLECTE` de `last_obs`.
   - `anciennete_last` = (date_last – prise_service) / 365,25.
   - `age_service_method3` = `age_last - anciennete_last`.

5. **Sélection de la meilleure méthode**

- On essaie successivement, avec priorité :
  1. `age_service_method1` (première obs) si `age_min ≤ âge ≤ age_max` → `methode = "PREMIERE_OBS"`.
  2. Sinon, `age_service_method2` si valide → `methode = "OBS_MEDIANE"`.
  3. Sinon, `age_service_method3` si valide → `methode = "DERNIERE_OBS"`.

- **Dernier recours** :
  - Si aucune des trois n’est dans `[age_min ; age_max]` :
    - on considère les trois candidats :
      - `("PREMIERE_OBS_INVALIDE", age_service_method1)`,
      - `("OBS_MEDIANE_INVALIDE", age_service_method2)`,
      - `("DERNIERE_OBS_INVALIDE", age_service_method3)`,
    - on choisit celle avec la valeur **la plus élevée** (`max(candidates, key=lambda x: x[0])`),
    - généralement, on retient la **moins négative** / la plus proche de la plage plausible.

6. **Propagation au groupe**
- La valeur choisie `age_service_final` est affectée à **toutes les lignes** du matricule :
  - `AGE_AU_SERVICE` = `age_service_final`,
  - `AGE_SERVICE_METHODE` = méthode retenue (`PREMIERE_OBS`, `OBS_MEDIANE`, etc.).
- `AGE_AU_SERVICE_VALIDE` :
  - = `age_service_final` si dans `[age_min ; age_max]`,
  - sinon `NaN`.

**Résultat :**
- Une **unique valeur d’âge au service par matricule** (`AGE_AU_SERVICE`), plus ou moins fiable selon la méthode.
- Une version filtrée sur la plage valide : `AGE_AU_SERVICE_VALIDE`.
- Une traçabilité de la méthode via `AGE_SERVICE_METHODE`.

---

1.4 Diagnostics et statistiques

Après l’application du calcul robuste, la fonction :

- Dresse un **bilan des méthodes utilisées** :
  - compte le nombre de matricules par `AGE_SERVICE_METHODE`,
  - affiche pourcentage et statut :
    - `✅` pour les méthodes valides (`PREMIERE_OBS`, `OBS_MEDIANE`, `DERNIERE_OBS`),
    - `⚠️` pour les méthodes marquées `*_INVALIDE`.

- Compte les **âges de service négatifs** :
  - nombre d’observations et de matricules concernés,
  - affiche des **causes possibles** :
    - prise de service trop ancienne,
    - âge imputé trop faible,
    - confusion avec la date de naissance,
    - problème de conversion de dates (Excel).

- Produit des **statistiques globales** :
  - % d’observations avec `AGE_AU_SERVICE` calculé,
  - % d’observations avec `AGE_AU_SERVICE_VALIDE`,
  - % de matricules ayant un âge calculé et un âge valide.

- Affiche, si `AGE_AU_SERVICE_VALIDE` non vide :
  - min, Q25, médiane, moyenne, Q75, max de `AGE_AU_SERVICE_VALIDE`.

---

2. `ageps_coherence_non_temporelle(df)`

Vérifie la **cohérence non temporelle** de l’âge de prise de service, observation par observation.

**Étapes :**

1. **Sélection de la colonne d’âge au service** (`age_ps_col`) :
   - première trouvée parmi :
     - `"AGE_AU_SERVICE"`,
     - `"AGE_PRISERVICE"`.
   - sinon → création de `AGE_PS_TMP` remplie de `NaN`.

2. Conversion de `age_ps_col` en numérique.

3. **FLAG 1 – `FLAG_AGE_PS_NEGATIF`**
   - Condition :
     - `age_ps_col < 0`.
   - Interprétation :
     - âge de prise de service **négatif** → toujours incohérent.

4. **FLAG 2 – `FLAG_AGE_PS_GE_AGE`**
   - Recherche d’une colonne d’âge actuel (`age_col`) :
     - `"AGE_IMPUTE"` puis `"AGE"`.
   - Si trouvée, convertie en numérique.
   - Condition :
     - `age_ps_col >= age_col`,
     - `age_ps_col` non nul,
     - `age_col` non nul.
   - Interprétation :
     - âge au service **supérieur ou égal** à l’âge actuel, ce qui n’est pas possible
       (l’entrée en service doit être plus **jeune** que l’âge courant).

   - Si aucune colonne d’âge actuel n’est trouvée :
     - `FLAG_AGE_PS_GE_AGE` est mis à 0 pour toutes les lignes.

**Résultat :**
- DataFrame enrichi avec :
  - `FLAG_AGE_PS_NEGATIF`,
  - `FLAG_AGE_PS_GE_AGE`.

---

3. `ageps_coherence_temporelle(df)`

Vérifie la **cohérence temporelle** de l’âge de prise de service pour un même individu, c’est-à-dire si cette valeur varie dans le temps.

**Pré-requis :**
- `DATE_REF` doit exister :
  - sinon, création via `_ensure_date_ref(df)`.
- Présence de `MATRICULE` + `DATE_REF` vérifiée via `_need_keys(df)`.

**Étapes :**

1. **Sélection de la colonne d’âge au service** (`age_ps_col`) :
   - première trouvée parmi :
     - `"AGE_AU_SERVICE"`,
     - `"AGE_PRISERVICE"`.
   - Si aucune → `ValueError`.

2. Copie du DataFrame et tri par `(MATRICULE, DATE_REF)`.

3. **Comparaison temporelle** :
   - `'_AGEPS_PREV'` = âge au service de la période précédente (`groupby().shift(1)`).
   - `FLAG_AGE_PS_VARIATION` :
     - vaut 1 si :
       - `age_ps_col` ≠ `_AGEPS_PREV`,
       - `age_ps_col` non nul,
       - `_AGEPS_PREV` non nul.
     - 0 sinon.

   → Tout changement de valeur dans le temps pour un même matricule est flaggé.

4. Suppression de la colonne temporaire `'_AGEPS_PREV'`.

**Résultat :**
- DataFrame enrichi avec :
  - `FLAG_AGE_PS_VARIATION`.



In [8]:
def build_age_service_stable(df, matricule_col="MATRICULE", 
                             start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
                             collect_col="DATE_COLLECTE",
                             age_src_priority=("AGE_IMPUTE", "AGE"),
                             age_min=14, age_max=60):
    """
    Construire l'âge au service de manière STABLE (une seule valeur par matricule)
    VERSION ROBUSTE avec détection et correction des âges négatifs
    
    MÉTHODE 2 : Calcul à partir de AGE_IMPUTE à la première observation
    avec stratégies alternatives en cas de problème
    
    Principe:
        AGE_AU_SERVICE = AGE_première_observation - ANCIENNETÉ_première_observation
    
    Si le résultat est négatif ou hors limites, on essaie avec :
        - L'observation médiane
        - La dernière observation
        - La moins négative en dernier recours
    
    Args:
        df: DataFrame source
        matricule_col: Colonne d'identifiant
        start_col_corrected: Colonne de date de prise de service
        collect_col: Colonne de date de collecte
        age_src_priority: Tuple de colonnes d'âge à essayer
        age_min: Âge minimum valide (défaut: 14)
        age_max: Âge maximum valide (défaut: 60)
    
    Returns:
        DataFrame avec colonnes ajoutées:
        - AGE_AU_SERVICE : Âge calculé (stable par matricule)
        - AGE_AU_SERVICE_VALIDE : Âge validé (entre age_min et age_max)
        - AGE_SERVICE_METHODE : Méthode utilisée pour le calcul
    
    Causes possibles d'âges négatifs:
        1. Date de prise de service erronée (trop ancienne)
        2. Âge imputé trop faible
        3. Confusion entre date de naissance et date de prise de service
        4. Erreur de conversion de dates (format Excel)
    """
    df = df.copy()
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    
    # Corriger la date de prise de service si nécessaire
    if start_col_corrected not in df.columns:
        df["PRISE_DE_SERVICE_CLEAN"] = _clean_dates_generic(df["PRISE DE SERVICE"])
        panel_sorted = df.sort_values([matricule_col, collect_col], ascending=[True, False])
        ps_corr = (panel_sorted.drop_duplicates(subset=[matricule_col], keep="first")
                   [[matricule_col, "PRISE_DE_SERVICE_CLEAN"]]
                   .rename(columns={"PRISE_DE_SERVICE_CLEAN": start_col_corrected}))
        df = df.drop(columns=[start_col_corrected], errors="ignore").merge(
            ps_corr, on=matricule_col, how="left", validate="many_to_one"
        )
    
    df[start_col_corrected] = _to_datetime_mixed(df[start_col_corrected])
    
    # Trouver la colonne d'âge disponible
    age_src = None
    for col in age_src_priority:
        if col in df.columns:
            age_src = col
            print(f"   → Utilisation de '{col}' pour le calcul")
            break
    
    if age_src is None:
        raise ValueError("❌ Aucune colonne d'âge trouvée")
    
    print(f"   → Calcul de l'âge de service STABLE (version robuste avec détection des négatifs)...")
    
    # Trier par matricule et date
    df = df.sort_values([matricule_col, collect_col]).copy()
    
    # ============================================================
    # CALCUL ROBUSTE AVEC GESTION DES CAS PROBLÉMATIQUES
    # ============================================================
    def _calc_age_service_robust(g):
        """
        Calcul robuste avec plusieurs tentatives :
        1. Essayer avec la première observation
        2. Si négatif : essayer avec l'observation médiane
        3. Si toujours négatif : essayer avec la dernière observation
        4. En dernier recours : prendre la moins négative
        """
        g = g.copy()
        
        # Filtrer les observations valides
        valides = g[
            (g[age_src].notna()) & 
            (g[start_col_corrected].notna()) &
            (g[collect_col].notna())
        ]
        
        if valides.empty:
            g["AGE_AU_SERVICE"] = pd.NA
            g["AGE_AU_SERVICE_VALIDE"] = pd.NA
            g["AGE_SERVICE_METHODE"] = "AUCUNE_DONNEE"
            return g
        
        # Récupérer la date de prise de service (unique par matricule)
        prise_service = valides.iloc[0][start_col_corrected]
        
        # ============================================================
        # MÉTHODE 1 : Première observation (méthode standard)
        # ============================================================
        first_obs = valides.iloc[0]
        age_first = first_obs[age_src]
        date_first = first_obs[collect_col]
        anciennete_first = (date_first - prise_service).days / 365.25
        age_service_method1 = age_first - anciennete_first
        
        # ============================================================
        # MÉTHODE 2 : Observation médiane (si méthode 1 échoue)
        # ============================================================
        mid_idx = len(valides) // 2
        mid_obs = valides.iloc[mid_idx]
        age_mid = mid_obs[age_src]
        date_mid = mid_obs[collect_col]
        anciennete_mid = (date_mid - prise_service).days / 365.25
        age_service_method2 = age_mid - anciennete_mid
        
        # ============================================================
        # MÉTHODE 3 : Dernière observation (dernier recours)
        # ============================================================
        last_obs = valides.iloc[-1]
        age_last = last_obs[age_src]
        date_last = last_obs[collect_col]
        anciennete_last = (date_last - prise_service).days / 365.25
        age_service_method3 = age_last - anciennete_last
        
        # ============================================================
        # SÉLECTION DE LA MEILLEURE MÉTHODE
        # ============================================================
        
        # Priorité 1 : Méthode 1 si valide
        if age_min <= age_service_method1 <= age_max:
            age_service_final = age_service_method1
            methode = "PREMIERE_OBS"
        
        # Priorité 2 : Méthode 2 si valide
        elif age_min <= age_service_method2 <= age_max:
            age_service_final = age_service_method2
            methode = "OBS_MEDIANE"
        
        # Priorité 3 : Méthode 3 si valide
        elif age_min <= age_service_method3 <= age_max:
            age_service_final = age_service_method3
            methode = "DERNIERE_OBS"
        
        # Dernier recours : prendre la moins négative (ou la plus proche de l'intervalle)
        else:
            candidates = [
                (age_service_method1, "PREMIERE_OBS_INVALIDE"),
                (age_service_method2, "OBS_MEDIANE_INVALIDE"),
                (age_service_method3, "DERNIERE_OBS_INVALIDE")
            ]
            # Prendre la valeur la plus proche de l'intervalle valide
            age_service_final, methode = max(candidates, key=lambda x: x[0])
        
        # Propager à tout le groupe
        g["AGE_AU_SERVICE"] = age_service_final
        g["AGE_SERVICE_METHODE"] = methode
        
        # Validation finale
        if age_min <= age_service_final <= age_max:
            g["AGE_AU_SERVICE_VALIDE"] = age_service_final
        else:
            g["AGE_AU_SERVICE_VALIDE"] = pd.NA
        
        return g
    
    # Appliquer le calcul robuste
    df = df.groupby(matricule_col, group_keys=False).apply(_calc_age_service_robust)
    
    # ============================================================
    # DIAGNOSTIC DES PROBLÈMES
    # ============================================================
    print(f"\n   📊 Diagnostic des méthodes utilisées :")
    
    methode_counts = df.groupby(matricule_col)["AGE_SERVICE_METHODE"].first().value_counts()
    for methode, count in methode_counts.items():
        pct = (count / df[matricule_col].nunique()) * 100
        status = "✅" if "INVALIDE" not in methode else "⚠️"
        print(f"      {status} {methode:30s} : {count:6,} matricules ({pct:5.1f}%)")
    
    # Compter les âges négatifs
    nb_negatifs = (df["AGE_AU_SERVICE"] < 0).sum()
    nb_mat_negatifs = df[df["AGE_AU_SERVICE"] < 0][matricule_col].nunique()
    
    if nb_negatifs > 0:
        pct_neg_obs = (nb_negatifs / len(df)) * 100
        pct_neg_mat = (nb_mat_negatifs / df[matricule_col].nunique()) * 100
        
        print(f"\n   ⚠️  ATTENTION : Âges de service négatifs détectés !")
        print(f"      • Observations : {nb_negatifs:,} ({pct_neg_obs:.2f}%)")
        print(f"      • Matricules : {nb_mat_negatifs:,} ({pct_neg_mat:.2f}%)")
        print(f"\n   💡 Causes possibles :")
        print(f"      1. Date de prise de service erronée (trop ancienne)")
        print(f"      2. Âge imputé trop faible")
        print(f"      3. Confusion entre date de naissance et date de prise de service")
        print(f"      4. Erreur de conversion de dates (format Excel)")
    else:
        print(f"\n   ✅ Aucun âge de service négatif détecté !")
    
    # Statistiques générales
    nb_total = len(df)
    nb_calcule = df["AGE_AU_SERVICE"].notna().sum()
    nb_valide = df["AGE_AU_SERVICE_VALIDE"].notna().sum()
    
    nb_matricules = df[matricule_col].nunique()
    nb_mat_calcule = df[df["AGE_AU_SERVICE"].notna()][matricule_col].nunique()
    nb_mat_valide = df[df["AGE_AU_SERVICE_VALIDE"].notna()][matricule_col].nunique()
    
    print(f"\n   📊 Statistiques globales :")
    print(f"      • Observations calculées : {nb_calcule:,} / {nb_total:,} ({nb_calcule/nb_total*100:.1f}%)")
    print(f"      • Observations valides : {nb_valide:,} / {nb_total:,} ({nb_valide/nb_total*100:.1f}%)")
    print(f"      • Matricules calculés : {nb_mat_calcule:,} / {nb_matricules:,} ({nb_mat_calcule/nb_matricules*100:.1f}%)")
    print(f"      • Matricules valides : {nb_mat_valide:,} / {nb_matricules:,} ({nb_mat_valide/nb_matricules*100:.1f}%)")
    
    # Distribution
    if nb_valide > 0:
        print(f"\n   📈 Distribution de l'âge de service valide :")
        stats = df["AGE_AU_SERVICE_VALIDE"].describe()
        print(f"      • Min     : {stats['min']:.1f} ans")
        print(f"      • Q25     : {stats['25%']:.1f} ans")
        print(f"      • Médiane : {stats['50%']:.1f} ans")
        print(f"      • Moyenne : {stats['mean']:.1f} ans")
        print(f"      • Q75     : {stats['75%']:.1f} ans")
        print(f"      • Max     : {stats['max']:.1f} ans")
    
    return df


def ageps_coherence_non_temporelle(df):
    """
    Vérifier la cohérence NON temporelle de l'âge de prise de service
    
    Contrôles effectués:
    1. FLAG_AGE_PS_NEGATIF : Âge de service négatif → erreur grave
    2. FLAG_AGE_PS_GE_AGE : Âge de service >= âge actuel → impossible
    
    Args:
        df: DataFrame à contrôler
    
    Returns:
        DataFrame avec flags ajoutés
    
    Exemples:
        AGE_AU_SERVICE = -5 ans → FLAG_AGE_PS_NEGATIF = 1
        AGE_AU_SERVICE = 45 ans, AGE = 40 ans → FLAG_AGE_PS_GE_AGE = 1
    """
    df = df.copy()
    
    # Détecter la colonne d'âge de prise de service
    age_ps_col = next((c for c in ['AGE_AU_SERVICE', 'AGE_PRISERVICE'] if c in df.columns), None)
    if age_ps_col is None:
        df['AGE_PS_TMP'] = np.nan
        age_ps_col = 'AGE_PS_TMP'
    
    df[age_ps_col] = pd.to_numeric(df[age_ps_col], errors='coerce')
    
    # FLAG 1 : Âge de service négatif (NOUVEAU)
    df['FLAG_AGE_PS_NEGATIF'] = (df[age_ps_col] < 0).astype('Int8')
    
    # FLAG 2 : Âge de service >= âge actuel (impossible biologiquement)
    age_col = next((c for c in ['AGE_IMPUTE', 'AGE'] if c in df.columns), None)
    if age_col:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        df['FLAG_AGE_PS_GE_AGE'] = ((df[age_ps_col] >= df[age_col]) & 
                                     df[age_ps_col].notna() & df[age_col].notna()).astype('Int8')
    else:
        df['FLAG_AGE_PS_GE_AGE'] = pd.Series(0, index=df.index, dtype='Int8')
    
    return df


def ageps_coherence_temporelle(df):
    """
    Vérifier la cohérence TEMPORELLE de l'âge de prise de service
    
    Contrôle effectué:
    - FLAG_AGE_PS_VARIATION : Âge de service qui varie dans le temps
    
    Args:
        df: DataFrame à contrôler
    
    Returns:
        DataFrame avec flag ajouté
    
    Principe:
        L'âge de prise de service est un événement HISTORIQUE unique.
        Il ne devrait JAMAIS varier pour un même matricule.
    
    Note:
        Avec la nouvelle méthode de calcul stable, ce flag devrait être à 0.
    """
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    
    print("   → Analyse des variations temporelles...")
    
    # Détecter la colonne d'âge de prise de service
    age_ps_col = next((c for c in ['AGE_AU_SERVICE', 'AGE_PRISERVICE'] if c in df.columns), None)
    if age_ps_col is None:
        raise ValueError("Aucune colonne d'âge de prise de service trouvée.")
    
    df = df.copy()
    df = df.sort_values(['MATRICULE', 'DATE_REF']).copy()
    
    # Calcul de la valeur précédente
    df['_AGEPS_PREV'] = df.groupby('MATRICULE')[age_ps_col].shift(1)
    
    # FLAG : Variation détectée
    df['FLAG_AGE_PS_VARIATION'] = (
        (df[age_ps_col] != df['_AGEPS_PREV']) &
        df[age_ps_col].notna() & 
        df['_AGEPS_PREV'].notna()
    ).astype('Int8')
    
    # Nettoyage
    df = df.drop(columns=['_AGEPS_PREV'], errors='ignore')
    
    return df

# ANCIENNETÉ

Cette section gère :
- la **construction d’une ancienneté simple** en années (`ANCIENNETE`),
- les **contrôles non temporels** de cohérence de l’ancienneté,
- les **contrôles temporels** de l’évolution de l’ancienneté dans le temps.

Elle introduit notamment les flags :
- `FLAG_ANC_NEGATIVE` : ancienneté négative,
- `FLAG_ANC_IDENTITE_IRR` : identité âge ≈ âge au service + ancienneté non respectée,
- `FLAG_ANC_BAISSE` : ancienneté qui diminue dans le temps,
- `FLAG_ANC_VITESSE_IRR` : ancienneté qui augmente trop vite par rapport au temps écoulé.

---

1. `build_anciennete_simple(...)`

Construit une **ancienneté simple en années** à partir de la date de prise de service et de la date de collecte.

**Paramètres principaux :**
- `df` : DataFrame source.
- `matricule_col` : identifiant de l’individu (non utilisé directement dans cette fonction, mais important en aval).
- `collect_col` : date de collecte (par défaut `"DATE_COLLECTE"`).
- `start_service_col` : date de prise de service corrigée (par défaut `"PRISE_DE_SERVICE_CORRIGEE"`).

**Étapes de traitement :**
1. Conversion de `DATE_COLLECTE` et `PRISE_DE_SERVICE_CORRIGEE` en datetime (`errors="coerce"`).
2. Création d’un masque `mask` :
   - `collect_col` non nul,
   - `start_service_col` non nul.
3. Initialisation de `ANCIENNETE` à `NA` (type `Float64`).
4. Pour les lignes où `mask = True` :
   - `delta_days` = (DATE_COLLECTE – PRISE_DE_SERVICE_CORRIGEE) en jours,
   - `ANCIENNETE` = `delta_days / 365.25`, **arrondi à 2 décimales**.

**Résultat :**
- `ANCIENNETE` : temps écoulé entre la prise de service et la collecte, en années, avec décimales.

---

2. `anc_coherence_non_temporelle(df)`

Vérifie la **cohérence non temporelle** de la variable d’ancienneté, observation par observation.

2.1 Sélection / création de la colonne d’ancienneté

- `anc_col` :
  - première trouvée parmi :
    - `"ANCIENNETE"`,
    - `"ANCIENNETE_AN_PERS"`.
  - sinon, création d’une colonne temporaire `ANCIENNETE_TMP` remplie de `NaN`.

- Conversion de `anc_col` en numérique (`to_numeric(..., errors='coerce')`).

2.2 Flags non temporels

- `FLAG_ANC_NEGATIVE`  
  - Condition :
    - `anc_col < 0`.
  - Interprétation :
    - ancienneté **négative** → incohérence (prise de service postérieure à la collecte, ou dates inversées).

---

2.3 Vérification de l’identité âge / âge au service / ancienneté

On suppose la relation théorique :

> `AGE ≈ AGE_AU_SERVICE + ANCIENNETE` (à ± 1 an près)

- `age_col` :
  - première trouvée parmi :
    - `"AGE_IMPUTE"`,
    - `"AGE"`.
- `age_ps_col` (âge à la prise de service) :
  - première trouvée parmi :
    - `"AGE_AU_SERVICE"`,
    - `"AGE_PRISERVICE"`.

Si `age_col` et `age_ps_col` existent tous les deux :
- conversion en numérique,
- calcul de :

  - `FLAG_ANC_IDENTITE_IRR` =
    - 1 si  
      `| age_col – (age_ps_col + anc_col) | > 1.0`  
    - 0 sinon.

Interprétation :
- si l’**identité âge ≈ âge au service + ancienneté** n’est pas respectée à ±1 an,
- cela signale une incohérence entre :
  - l’âge actuel,
  - l’âge au moment de la prise de service,
  - l’ancienneté calculée.

Si l’une des deux colonnes (`age_col`, `age_ps_col`) est absente :
- `FLAG_ANC_IDENTITE_IRR` est mis à 0 pour toutes les lignes.

2.4 Colonne de traçabilité

- `_ANC_COL` :
  - stocke le nom effectif de la colonne d’ancienneté utilisée,
  - utile pour d’autres traitements éventuels.

**Résultat :**
- DataFrame enrichi avec :
  - `FLAG_ANC_NEGATIVE`,
  - `FLAG_ANC_IDENTITE_IRR`,
  - `_ANC_COL`.

---

3. `anc_coherence_temporelle(df, max_drift=0.25)`

Vérifie la **cohérence temporelle** de l’ancienneté dans le temps pour chaque matricule, en tenant compte du temps écoulé entre deux `DATE_REF`.

**Paramètres :**
- `df` : DataFrame à contrôler.
- `max_drift` : dérive maximale autorisée (en années) **au-delà** du temps écoulé (par défaut 0,25 ≈ 3 mois).

3.1 Pré-requis

- `DATE_REF` doit exister :
  - sinon, création via `_ensure_date_ref(df)`.
- Présence de `MATRICULE` + `DATE_REF` vérifiée via `_need_keys(df)`.

3.2 Sélection de la colonne d’ancienneté

- `anc_col` :
  - première trouvée parmi :
    - `"ANCIENNETE"`,
    - `"ANCIENNETE_AN_PERS"`.
  - Si aucune → `ValueError`.

3.3 Calcul du delta temporel (si absent)

- Tri par `(MATRICULE, DATE_REF)`.
- Si `'_YEAR_DELTA'` n’existe pas encore :
  - `'_DATE_PREV'` = `DATE_REF` de la période précédente (via `groupby().shift(1)`),
  - `'_YEAR_DELTA'` = (DATE_REF – `_DATE_PREV`) / 365,25,
  - valeur mise à 0 pour la première observation de chaque matricule.

`_YEAR_DELTA` représente donc le **temps écoulé** entre deux relevés successifs, en années.

3.4 Calcul des flags temporels

- `'_ANC_PREV'` = ancienneté à la période précédente (via `groupby().shift(1)`).

**FLAG 1 : `FLAG_ANC_BAISSE`**

- Condition :
  - `anc_col < _ANC_PREV`,
  - `anc_col` non nul,
  - `_ANC_PREV` non nul.
- Interprétation :
  - **ancienneté qui diminue dans le temps** pour un même matricule,
  - situation impossible (on ne “perd” pas des années d’ancienneté),
  - signale une rupture dans les données ou des mises à jour erronées.

**FLAG 2 : `FLAG_ANC_VITESSE_IRR`**

- Condition :
  - `(anc_col – _ANC_PREV) > (_YEAR_DELTA + max_drift)`,
  - `anc_col` non nul,
  - `_ANC_PREV` non nul.
- Interprétation :
  - l’ancienneté augmente **plus vite** que :
    - le temps écoulé (`_YEAR_DELTA`)  
    + la marge de tolérance (`max_drift`, ex. 0,25 an ≈ 3 mois),
  - signale une progression **trop rapide** de l’ancienneté (double comptage, changement de référence, erreurs de saisie…).

3.5 Nettoyage

- Suppression de la colonne temporaire `'_ANC_PREV'` (mais `_YEAR_DELTA` est laissé pour usage ultérieur).

**Résultat :**
- DataFrame enrichi avec :
  - `FLAG_ANC_BAISSE`,
  - `FLAG_ANC_VITESSE_IRR`,
  - et éventuellement `_DATE_PREV`, `_YEAR_DELTA` si non supprimés ailleurs.


In [9]:
def build_anciennete_simple(df, matricule_col="MATRICULE", collect_col="DATE_COLLECTE",
                            start_service_col="PRISE_DE_SERVICE_CORRIGEE"):
    """Construire une ancienneté simple en années"""
    df = df.copy()
    
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")
    df[start_service_col] = pd.to_datetime(df[start_service_col], errors="coerce")
    
    mask = df[collect_col].notna() & df[start_service_col].notna()
    df['ANCIENNETE'] = pd.Series(pd.NA, index=df.index, dtype='Float64')
    
    if mask.any():
        delta_days = (df.loc[mask, collect_col] - df.loc[mask, start_service_col]).dt.days
        df.loc[mask, 'ANCIENNETE'] = (delta_days / 365.25).round(2)
    
    return df


def anc_coherence_non_temporelle(df):
    """Vérifier la cohérence non temporelle de l'ancienneté"""
    df = df.copy()
    
    anc_col = next((c for c in ['ANCIENNETE', 'ANCIENNETE_AN_PERS'] if c in df.columns), None)
    if anc_col is None:
        df['ANCIENNETE_TMP'] = np.nan
        anc_col = 'ANCIENNETE_TMP'
    
    df[anc_col] = pd.to_numeric(df[anc_col], errors='coerce')
    df['FLAG_ANC_NEGATIVE'] = (df[anc_col] < 0).astype('Int8')

    age_col = next((c for c in ['AGE_IMPUTE', 'AGE'] if c in df.columns), None)
    age_ps_col = next((c for c in ['AGE_AU_SERVICE', 'AGE_PRISERVICE'] if c in df.columns), None)
    
    if age_col and age_ps_col:
        df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
        df[age_ps_col] = pd.to_numeric(df[age_ps_col], errors='coerce')
        df['FLAG_ANC_IDENTITE_IRR'] = (np.abs(df[age_col] - (df[age_ps_col] + df[anc_col])) > 1.0).astype('Int8')
    else:
        df['FLAG_ANC_IDENTITE_IRR'] = pd.Series(0, index=df.index, dtype='Int8')

    df['_ANC_COL'] = anc_col
    return df


def anc_coherence_temporelle(df, max_drift=0.25):
    """Vérifier la cohérence temporelle de l'ancienneté - VERSION OPTIMISÉE"""
    if 'DATE_REF' not in df.columns:
        df = _ensure_date_ref(df)
    _need_keys(df)
    
    print("   → Analyse des variations temporelles...")
    
    anc_col = next((c for c in ['ANCIENNETE', 'ANCIENNETE_AN_PERS'] if c in df.columns), None)
    if anc_col is None:
        raise ValueError("Aucune colonne d'ancienneté trouvée.")
    
    df = df.copy()
    df = df.sort_values(['MATRICULE', 'DATE_REF']).copy()
    
    # Calcul du delta temporel si pas déjà fait
    if '_YEAR_DELTA' not in df.columns:
        df['_DATE_PREV'] = df.groupby('MATRICULE')['DATE_REF'].shift(1)
        df['_YEAR_DELTA'] = ((df['DATE_REF'] - df['_DATE_PREV']).dt.days / 365.25).fillna(0)
    
    # Calcul des flags
    df['_ANC_PREV'] = df.groupby('MATRICULE')[anc_col].shift(1)
    df['FLAG_ANC_BAISSE'] = (
        (df[anc_col] < df['_ANC_PREV']) & 
        df[anc_col].notna() & 
        df['_ANC_PREV'].notna()
    ).astype('Int8')
    
    df['FLAG_ANC_VITESSE_IRR'] = (
        ((df[anc_col] - df['_ANC_PREV']) > (df['_YEAR_DELTA'] + max_drift)) &
        df[anc_col].notna() & 
        df['_ANC_PREV'].notna()
    ).astype('Int8')
    
    df = df.drop(columns=['_ANC_PREV'], errors='ignore')
    
    return df



# CHARGEMENT DE LA BASE 

In [10]:
def charger_donnees_s3():
    """Charger les données depuis S3/MinIO"""
    s3_client = boto3.client(
        "s3",
        endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False
    )

    bucket_name = "admindataanstat"
    object_key = "Solde/panel_solde_df_1_code.parquet"

    obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

    return panel_solde_df



# FONCTIONS MODULAIRES POUR EXÉCUTION PAS À PAS

Ce bloc décrit le **scénario complet de traitement** du panel de solde, organisé en 7 grandes étapes après initialisation :

0. Initialisation et chargement  
1. Situation matrimoniale  
2. Nombre d’enfants  
3. Sexe  
4. Âge  
5. Âge de prise de service  
6. Ancienneté  
7. Rapport final consolidé

Chaque étape :
- transforme le `DataFrame` principal,
- ajoute des variables/flags,
- appelle `ControleQualite.controler_variable` pour produire des exports détaillés.

---

**`etape_0_initialisation()`**

**Rôle :**  
Initialiser l’environnement, charger les données brutes et produire une **description rapide des variables**.

**Actions principales :**
- Affiche un en-tête « ÉTAPE 0 : INITIALISATION ».
- Crée une instance de `ControleQualite` avec `output_dir="controles_qualite"`.
- Appelle `charger_donnees_s3()` pour charger la base principale dans `df`.
- Affiche les informations de base :
  - nombre total d’observations,
  - nombre de `MATRICULE` uniques,
  - période couverte (`DATE_COLLECTE.min()` à `DATE_COLLECTE.max()`),
  - nombre de variables.

**Description des variables :**
- Pour chaque colonne :
  - type (`dtype`),
  - nombre de non-nuls + pourcentage,
  - un exemple de valeur (première valeur non nulle, tronquée à 50 caractères).
- Construit un `DataFrame` `df_info` et le `display()`.

**Sorties :**
- `df` : données chargées et inspectées.
- `controle` : instance de `ControleQualite` prête pour les étapes suivantes.

---

**`etape_1_situation_matrimoniale(df, controle)`**

**Rôle :**  
Traiter et contrôler la **situation matrimoniale**, avec cohérences non temporelles et temporelles.

**Sous-étapes :**
1. Affichage de l’en-tête de l’étape.
2. `build_situation_matrimoniale(df)` :
   - recodage des modalités de situation matrimoniale,
   - retour de `df` enrichi + rapport `rep_sitmat`.
3. `sitmat_coherence_non_temporelle(df)` :
   - normalisation (`_SITMAT_NORM`, `_SITMAT_RECODE`),
   - flag :
     - `FLAG_SITMAT_AGE` : marié(e) avec âge < 18 ans.
4. `sitmat_coherence_temporelle(df)` :
   - détection des changements de statut d’un mois à l’autre :
     - `FLAG_SITMAT_VARIATION`.

**Contrôle qualité :**
- Appel à `controle.controler_variable` sur `"SITUATION_MATRIMONIALE_RECODE"` avec les flags suivants :

  - `FLAG_SITMAT_AGE`  
    - Description : marié(e) avant 18 ans → incohérence.  
    - Colonnes export : `['_SITMAT_RECODE', 'AGE_IMPUTE', 'AGE', 'NOM', 'SEXE']`.

  - `FLAG_SITMAT_VARIATION`  
    - Description : changement de situation matrimoniale entre deux mois consécutifs.  
    - Colonnes export : `['_SITMAT_RECODE', 'NOM']`.

  - `FLAG_SITMAT_VARIATIONS`  
    - Description : plus d’un changement la même année → instabilité suspecte.  
    - Colonnes export : `['_SITMAT_RECODE', 'NOM']`.  
    - (À noter : si le flag n’est plus calculé dans les fonctions amont, ce bloc ne produira plus de sortie pour ce flag.)

**Sorties :**
- `df` : enrichi avec variables/flags de situation matrimoniale.
- `rep_sitmat` : rapport de recodage (distribution des modalités recodées).

---

**`etape_2_nombre_enfants(df, controle)`**

**Rôle :**  
Traiter et contrôler le **nombre d’enfants** (construction + cohérences).

**Sous-étapes :**
1. Affichage de l’en-tête de l’étape.
2. `build_nombre_enfants(df)` :
   - création de `NBRE_ENFTS_IMPUTE` à partir de la colonne brute (ex. `"NOMBRE ENFANT"`),
   - imputation des manquants (valeur par défaut),
   - rapport `rep_enfants` (distribution).
3. `enfants_coherence_non_temporelle(df)` :
   - détection de colonnes d’âge et d’enfants,
   - flags :
     - `FLAG_ENFANTS_AGE` : individu < 15 ans avec ≥ 1 enfant,
     - `FLAG_ENFANTS_FERT_IRR` : nombre d’enfants > (âge – 12).
4. `enfants_coherence_temporelle(df)` :
   - reconstitution de la trajectoire par `MATRICULE`,
   - `FLAG_ENFANTS_DESCENTE` : nombre d’enfants qui diminue d’un mois à l’autre.

**Contrôle qualité :**
- Sur la variable `"NBRE_ENFTS_IMPUTE"` :

  - `FLAG_ENFANTS_AGE`  
    - Description : enfant(s) alors que l’individu a < 15 ans → incohérence.  
    - Colonnes export : `['NBRE_ENFTS_IMPUTE', 'AGE_IMPUTE', 'AGE', 'NOM', 'SEXE']`.

  - `FLAG_ENFANTS_FERT_IRR`  
    - Description : nombre d’enfants > (âge - 12) → fertilité irréaliste.  
    - Colonnes export : `['NBRE_ENFTS_IMPUTE', 'AGE_IMPUTE', 'NOM', 'SEXE']`.

  - `FLAG_ENFANTS_DESCENTE`  
    - Description : nombre d’enfants qui diminue dans le temps → incohérence.  
    - Colonnes export : `['NBRE_ENFTS_IMPUTE', 'NOM']`.

**Sorties :**
- `df` : enrichi avec `NBRE_ENFTS_IMPUTE` et ses flags.
- `rep_enfants` : rapport sur la distribution des enfants.

---

**`etape_3_sexe(df, controle)`**

**Rôle :**  
Nettoyer, imputer et contrôler le **sexe** des individus.

**Sous-étapes :**
1. Affichage de l’en-tête de l’étape.
2. `build_sexe_clean(df)` :
   - stabilise le sexe au niveau du matricule (dernière valeur observée),
   - crée `SEXE_CLEAN`.
3. `imputer_sexe(df, debug=False)` :
   - impute les valeurs manquantes à partir du mode `(ANNEE_COLLECTE, MOIS_COLLECTE)`,
   - produit `SEXE_IMPUTE`.
4. `sexe_coherence_temporelle(df)` :
   - normalise via `_SEXE_NORM` (M/F),
   - détecte les variations :
     - `FLAG_SEXE_VARIATION` : changement de sexe dans le temps pour un même individu.

**Contrôle qualité :**
- Sur `"SEXE_IMPUTE"` :

  - `FLAG_SEXE_VARIATION`  
    - Description : sexe qui varie dans le temps → incohérence grave.  
    - Colonnes export : `['SEXE', 'SEXE_CLEAN', 'SEXE_IMPUTE', '_SEXE_NORM', 'NOM']`.

**Sorties :**
- `df` : enrichi avec `SEXE_CLEAN`, `SEXE_IMPUTE`, `_SEXE_NORM`, `FLAG_SEXE_VARIATION`.

---

**`etape_4_age(df, controle)`**

**Rôle :**  
Construire, imputer et contrôler l’**âge** des individus.

**Sous-étapes :**
1. Affichage de l’en-tête de l’étape.
2. `build_age_valide(df)` :
   - nettoyage de la date de naissance,
   - calcul de `AGE`,
   - définition de `AGE_VALIDE` (dans un intervalle plausible),
   - imputation temporellement cohérente `AGE_IMPUTE`,
   - stabilisation éventuelle `AGE_IMPUTE_STABLE`.
3. `age_coherence_temporelle(df)` :
   - reconstitution des trajectoires par `MATRICULE`,
   - calcul de `_YEAR_DELTA` (temps écoulé entre deux `DATE_REF`),
   - flags :
     - `FLAG_AGE_DECROIT` : âge qui diminue,
     - `FLAG_AGE_TROP_RAPIDE` : âge qui augmente trop vite par rapport au temps écoulé + tolérance.

**Contrôle qualité :**
- Sur `"AGE_IMPUTE"` :

  - `FLAG_AGE_DECROIT`  
    - Description : âge qui diminue dans le temps → incohérence.  
    - Colonnes export : `['AGE', 'AGE_IMPUTE', 'AGE_IMPUTE_STABLE', 'DATE NAISSANCE', 'NOM']`.

  - `FLAG_AGE_TROP_RAPIDE`  
    - Description : âge qui augmente trop vite (> temps écoulé + 1 an).  
    - Colonnes export : `['AGE', 'AGE_IMPUTE', '_YEAR_DELTA', 'DATE NAISSANCE', 'NOM']`.

**Sorties :**
- `df` : enrichi avec les variables d’âge et leurs flags.

---

**`etape_5_age_service(df, controle)`**

**Rôle :**  
Construire et contrôler l’**âge de prise de service** de manière robuste.

**Sous-étapes :**
1. Affichage de l’en-tête de l’étape.
2. `build_age_service_stable(df)` :
   - calcule un **âge au service STABLE** par matricule (`AGE_AU_SERVICE`),
   - gère les âges négatifs avec plusieurs stratégies (première obs, médiane, dernière obs),
   - renseigne `AGE_AU_SERVICE_VALIDE` si dans `[age_min ; age_max]`,
   - journalise les méthodes via `AGE_SERVICE_METHODE`,
   - affiche diagnostics et distribution.
3. `ageps_coherence_non_temporelle(df)` :
   - flags :
     - `FLAG_AGE_PS_NEGATIF` : âge au service négatif,
     - `FLAG_AGE_PS_GE_AGE` : âge au service ≥ âge actuel.
4. `ageps_coherence_temporelle(df)` :
   - `FLAG_AGE_PS_VARIATION` : variation de l’âge de prise de service dans le temps.

**Contrôle qualité :**
- Sur `"AGE_AU_SERVICE"` :

  - `FLAG_AGE_PS_NEGATIF`  
    - Description : âge de prise de service négatif → erreur grave de données.  
    - Colonnes export :  
      `['AGE_AU_SERVICE', 'AGE_IMPUTE', 'ANCIENNETE', 'PRISE DE SERVICE', 'PRISE_DE_SERVICE_CORRIGEE', 'DATE_COLLECTE', 'AGE_SERVICE_METHODE', 'NOM']`.

  - `FLAG_AGE_PS_VARIATION`  
    - Description : âge de prise de service qui varie dans le temps → incohérence (ne devrait plus arriver).  
    - Colonnes export :  
      `['AGE_AU_SERVICE', 'AGE_AU_SERVICE_VALIDE', 'AGE_SERVICE_METHODE', 'PRISE DE SERVICE', 'PRISE_DE_SERVICE_CORRIGEE', 'NOM']`.

  - `FLAG_AGE_PS_GE_AGE`  
    - Description : âge de prise de service ≥ âge actuel → impossible.  
    - Colonnes export :  
      `['AGE_AU_SERVICE', 'AGE_AU_SERVICE_VALIDE', 'AGE_IMPUTE', 'AGE', 'PRISE DE SERVICE', 'NOM']`.

**Sorties :**
- `df` : enrichi avec les variables d’âge au service et leurs flags.

---

**`etape_6_anciennete(df, controle)`**

**Rôle :**  
Calculer et contrôler l’**ancienneté** dans l’emploi.

**Sous-étapes :**
1. Affichage de l’en-tête de l’étape.
2. `build_anciennete_simple(df)` :
   - calcule `ANCIENNETE` = (DATE_COLLECTE – PRISE_DE_SERVICE_CORRIGEE) / 365,25.
3. `anc_coherence_non_temporelle(df)` :
   - sélection de la colonne d’ancienneté,
   - flags :
     - `FLAG_ANC_NEGATIVE` : ancienneté négative,
     - `FLAG_ANC_IDENTITE_IRR` : identité `âge ≈ âge_prise_service + ancienneté` non respectée (écart > 1 an).
4. `anc_coherence_temporelle(df)` :
   - trajectoire d’ancienneté par `MATRICULE`,
   - réutilisation de `_YEAR_DELTA` si présent ou calcul nouveau,
   - flags :
     - `FLAG_ANC_BAISSE` : ancienneté qui diminue,
     - `FLAG_ANC_VITESSE_IRR` : ancienneté qui augmente trop vite (> temps écoulé + 3 mois, via `max_drift=0.25`).

**Contrôle qualité :**
- Sur `"ANCIENNETE"` :

  - `FLAG_ANC_NEGATIVE`  
    - Description : ancienneté négative → incohérence.  
    - Colonnes export : `['ANCIENNETE', 'PRISE DE SERVICE', 'PRISE_DE_SERVICE_CORRIGEE', 'NOM']`.

  - `FLAG_ANC_IDENTITE_IRR`  
    - Description : identité âge = âge_prise_service + ancienneté non respectée (écart > 1 an).  
    - Colonnes export : `['ANCIENNETE', 'AGE_IMPUTE', 'AGE_AU_SERVICE', 'NOM']`.

  - `FLAG_ANC_BAISSE`  
    - Description : ancienneté qui diminue dans le temps → incohérence.  
    - Colonnes export : `['ANCIENNETE', 'PRISE DE SERVICE', 'NOM']`.

  - `FLAG_ANC_VITESSE_IRR`  
    - Description : ancienneté qui augmente trop vite (> temps écoulé + 3 mois).  
    - Colonnes export : `['ANCIENNETE', '_YEAR_DELTA', 'PRISE DE SERVICE', 'NOM']`.

**Sorties :**
- `df` : enrichi avec l’ancienneté et ses flags.

---

**`etape_7_rapport_final(controle)`**

**Rôle :**  
Générer le **rapport global consolidé** des contrôles de qualité.

**Actions :**
- Affiche l’en-tête « ÉTAPE 7 : RAPPORT FINAL ».
- Appelle `controle.generer_rapport_global()` :
  - construit un fichier Excel de synthèse (feuille “Synthèse”) listant :
    - variable,
    - flag,
    - description,
    - nb d’observations flaggées,
    - nb de matricules concernés,
    - pourcentages,
    - nom du fichier détaillé exporté.
- Affiche un récapitulatif :
  - chemin du dossier des fichiers de contrôle (`controle.output_dir`),
  - nombre total de **types de problèmes détectés** (`len(controle.problemes_detectes)`).

**Sorties :**
- Aucun nouveau `df` (on ne modifie plus les données),
- un rapport global Excel + tous les fichiers détaillés déjà générés par `ControleQualite`.

---

**Vue d’ensemble du pipeline**

En enchaînant les étapes `etape_0` à `etape_7`, on obtient :

- un **panel propre** avec :
  - variables recodées (situation matrimoniale, nombre d’enfants, sexe, âge, âge de service, ancienneté),
  - variables imputées / stabilisées,
  - flags de cohérence non temporelle et temporelle.

- un **système de reporting complet** :
  - exports détaillés par variable/flag,
  - rapport global consolidé,
  - possibilité d’inspecter le **contexte complet** des matricules problématiques.


In [11]:

def etape_0_initialisation():
    """ÉTAPE 0 : Initialisation et chargement des données"""
    print("="*80)
    print("  ÉTAPE 0 : INITIALISATION")
    print("="*80)
    
    # Initialiser le système de contrôle
    controle = ControleQualite(output_dir="controles_qualite")
    
    # Charger les données
    print("\n📂 Chargement des données...")
    df = charger_donnees_s3()
    print(f"✓ Données chargées : {len(df):,} observations")
    print(f"✓ Nombre de matricules uniques : {df['MATRICULE'].nunique():,}")
    print(f"✓ Période couverte : {df['DATE_COLLECTE'].min()} à {df['DATE_COLLECTE'].max()}")
    print(f"✓ Nombre de variables : {len(df.columns)}\n")
    
    # Afficher informations sur les variables
    print("="*80)
    print("  DESCRIPTION DES VARIABLES")
    print("="*80 + "\n")
    
    info_vars = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        nb_non_null = df[col].notna().sum()
        pct_non_null = (nb_non_null / len(df)) * 100
        exemple = df[col].dropna().iloc[0] if nb_non_null > 0 else "N/A"
        
        info_vars.append({
            'Variable': col,
            'Type': dtype,
            'Non-nulls': f"{nb_non_null:,} ({pct_non_null:.1f}%)",
            'Exemple': str(exemple)[:50]  # Limiter à 50 caractères
        })
    
    df_info = pd.DataFrame(info_vars)
    display(df_info)
    
    print("\n✅ Initialisation terminée\n")
    return df, controle


def etape_1_situation_matrimoniale(df, controle):
    """ÉTAPE 1 : Traitement et contrôle de la situation matrimoniale - VERSION OPTIMISÉE"""
    print("\n" + "="*80)
    print("  ÉTAPE 1 : SITUATION MATRIMONIALE")
    print("="*80 + "\n")
    
    print("🔄 Traitement en cours...")
    print("   → Recodage des modalités...")
    df, rep_sitmat = build_situation_matrimoniale(df)
    
    print("   → Vérification de la cohérence non temporelle...")
    df = sitmat_coherence_non_temporelle(df)
    
    print("   → Vérification de la cohérence temporelle...")
    df = sitmat_coherence_temporelle(df)
    
    print("\n📊 Contrôle qualité...\n")
    controle.controler_variable(
        df, 
        "SITUATION_MATRIMONIALE_RECODE",
        {
            'FLAG_SITMAT_AGE': {
                'description': 'Marié(e) avant 18 ans → incohérence',
                'colonnes_export': ['_SITMAT_RECODE', 'AGE_IMPUTE', 'AGE', 'NOM', 'SEXE']
            },
            'FLAG_SITMAT_VARIATION': {
                'description': 'Changement de situation matrimoniale entre deux mois consécutifs',
                'colonnes_export': ['_SITMAT_RECODE', 'NOM']
            },
            'FLAG_SITMAT_VARIATIONS': {
                'description': 'Plus d\'un changement la même année → instabilité suspecte',
                'colonnes_export': ['_SITMAT_RECODE', 'NOM']
            }
        }
    )
    
    print("✅ Étape 1 terminée\n")
    return df, rep_sitmat


def etape_2_nombre_enfants(df, controle):
    """ÉTAPE 2 : Traitement et contrôle du nombre d'enfants - VERSION OPTIMISÉE"""
    print("\n" + "="*80)
    print("  ÉTAPE 2 : NOMBRE D'ENFANTS")
    print("="*80 + "\n")
    
    print("🔄 Traitement en cours...")
    print("   → Imputation des valeurs manquantes...")
    df, rep_enfants = build_nombre_enfants(df)
    
    print("   → Vérification de la cohérence non temporelle...")
    df = enfants_coherence_non_temporelle(df)
    
    print("   → Vérification de la cohérence temporelle...")
    df = enfants_coherence_temporelle(df)
    
    print("\n📊 Contrôle qualité...\n")
    controle.controler_variable(
        df,
        "NBRE_ENFTS_IMPUTE",
        {
            'FLAG_ENFANTS_AGE': {
                'description': 'Enfant(s) alors que l\'individu a < 15 ans → incohérence',
                'colonnes_export': ['NBRE_ENFTS_IMPUTE', 'AGE_IMPUTE', 'AGE', 'NOM', 'SEXE']
            },
            'FLAG_ENFANTS_FERT_IRR': {
                'description': 'Nombre d\'enfants > (âge - 12) → fertilité irréaliste',
                'colonnes_export': ['NBRE_ENFTS_IMPUTE', 'AGE_IMPUTE', 'NOM', 'SEXE']
            },
            'FLAG_ENFANTS_DESCENTE': {
                'description': 'Nombre d\'enfants qui diminue dans le temps → incohérence',
                'colonnes_export': ['NBRE_ENFTS_IMPUTE', 'NOM']
            }
        }
    )
    
    print("✅ Étape 2 terminée\n")
    return df, rep_enfants


def etape_3_sexe(df, controle):
    """ÉTAPE 3 : Traitement et contrôle du sexe - VERSION OPTIMISÉE"""
    print("\n" + "="*80)
    print("  ÉTAPE 3 : SEXE")
    print("="*80 + "\n")
    
    print("🔄 Traitement en cours...")
    print("   → Nettoyage et stabilisation...")
    df = build_sexe_clean(df)
    
    print("   → Imputation des valeurs manquantes...")
    df = imputer_sexe(df, debug=False)
    
    print("   → Vérification de la cohérence temporelle...")
    df = sexe_coherence_temporelle(df)
    
    print("\n📊 Contrôle qualité...\n")
    controle.controler_variable(
        df,
        "SEXE_IMPUTE",
        {
            'FLAG_SEXE_VARIATION': {
                'description': 'Sexe qui varie dans le temps → incohérence grave',
                'colonnes_export': ['SEXE', 'SEXE_CLEAN', 'SEXE_IMPUTE', '_SEXE_NORM', 'NOM']
            }
        }
    )
    
    print("✅ Étape 3 terminée\n")
    return df


def etape_4_age(df, controle):
    """ÉTAPE 4 : Traitement et contrôle de l'âge - VERSION OPTIMISÉE"""
    print("\n" + "="*80)
    print("  ÉTAPE 4 : ÂGE")
    print("="*80 + "\n")
    
    print("🔄 Traitement en cours...")
    print("   → Calcul et imputation de l'âge...")
    df = build_age_valide(df)
    
    print("   → Vérification de la cohérence temporelle...")
    df = age_coherence_temporelle(df)
    
    print("\n📊 Contrôle qualité...\n")
    controle.controler_variable(
        df,
        "AGE_IMPUTE",
        {
            'FLAG_AGE_DECROIT': {
                'description': 'Âge qui diminue dans le temps → incohérence',
                'colonnes_export': ['AGE', 'AGE_IMPUTE', 'AGE_IMPUTE_STABLE', 'DATE NAISSANCE', 'NOM']
            },
            'FLAG_AGE_TROP_RAPIDE': {
                'description': 'Âge qui augmente trop vite (> temps écoulé + 1 an)',
                'colonnes_export': ['AGE', 'AGE_IMPUTE', '_YEAR_DELTA', 'DATE NAISSANCE', 'NOM']
            }
        }
    )
    
    print("✅ Étape 4 terminée\n")
    return df


def etape_5_age_service(df, controle):
    """ÉTAPE 5 : Traitement et contrôle de l'âge de prise de service - VERSION ROBUSTE"""
    print("\n" + "="*80)
    print("  ÉTAPE 5 : ÂGE DE PRISE DE SERVICE")
    print("="*80 + "\n")
    
    print("🔄 Traitement en cours...")
    print("   📌 MÉTHODE 2 ROBUSTE : Calcul avec détection des âges négatifs")
    df = build_age_service_stable(df)  # ← VERSION AMÉLIORÉE
    
    print("\n   → Vérification de la cohérence non temporelle...")
    df = ageps_coherence_non_temporelle(df)
    
    print("   → Vérification de la cohérence temporelle...")
    df = ageps_coherence_temporelle(df)
    
    print("\n📊 Contrôle qualité...\n")
    controle.controler_variable(
        df,
        "AGE_AU_SERVICE",
        {
            'FLAG_AGE_PS_NEGATIF': {  # ← NOUVEAU FLAG
                'description': 'Âge de prise de service NÉGATIF → erreur grave de données',
                'colonnes_export': ['AGE_AU_SERVICE', 'AGE_IMPUTE', 'ANCIENNETE', 
                                   'PRISE DE SERVICE', 'PRISE_DE_SERVICE_CORRIGEE', 
                                   'DATE_COLLECTE', 'AGE_SERVICE_METHODE', 'NOM']
            },
            'FLAG_AGE_PS_VARIATION': {
                'description': 'Âge de prise de service qui varie dans le temps → incohérence (NE DEVRAIT PLUS ARRIVER)',
                'colonnes_export': ['AGE_AU_SERVICE', 'AGE_AU_SERVICE_VALIDE', 'AGE_SERVICE_METHODE',
                                   'PRISE DE SERVICE', 'PRISE_DE_SERVICE_CORRIGEE', 'NOM']
            },
            'FLAG_AGE_PS_GE_AGE': {
                'description': 'Âge de prise de service >= âge actuel → impossible',
                'colonnes_export': ['AGE_AU_SERVICE', 'AGE_AU_SERVICE_VALIDE', 'AGE_IMPUTE', 
                                   'AGE', 'PRISE DE SERVICE', 'NOM']
            }
        }
    )
    
    print("✅ Étape 5 terminée\n")
    return df


def etape_6_anciennete(df, controle):
    """ÉTAPE 6 : Traitement et contrôle de l'ancienneté - VERSION OPTIMISÉE"""
    print("\n" + "="*80)
    print("  ÉTAPE 6 : ANCIENNETÉ")
    print("="*80 + "\n")
    
    print("🔄 Traitement en cours...")
    print("   → Calcul de l'ancienneté...")
    df = build_anciennete_simple(df)
    
    print("   → Vérification de la cohérence non temporelle...")
    df = anc_coherence_non_temporelle(df)
    
    print("   → Vérification de la cohérence temporelle...")
    df = anc_coherence_temporelle(df)
    
    print("\n📊 Contrôle qualité...\n")
    controle.controler_variable(
        df,
        "ANCIENNETE",
        {
            'FLAG_ANC_NEGATIVE': {
                'description': 'Ancienneté négative → incohérence',
                'colonnes_export': ['ANCIENNETE', 'PRISE DE SERVICE', 'PRISE_DE_SERVICE_CORRIGEE', 'NOM']
            },
            'FLAG_ANC_IDENTITE_IRR': {
                'description': 'Identité âge = âge_prise_service + ancienneté non respectée (écart > 1 an)',
                'colonnes_export': ['ANCIENNETE', 'AGE_IMPUTE', 'AGE_AU_SERVICE', 'NOM']
            },
            'FLAG_ANC_BAISSE': {
                'description': 'Ancienneté qui diminue dans le temps → incohérence',
                'colonnes_export': ['ANCIENNETE', 'PRISE DE SERVICE', 'NOM']
            },
            'FLAG_ANC_VITESSE_IRR': {
                'description': 'Ancienneté qui augmente trop vite (> temps écoulé + 3 mois)',
                'colonnes_export': ['ANCIENNETE', '_YEAR_DELTA', 'PRISE DE SERVICE', 'NOM']
            }
        }
    )
    
    print("✅ Étape 6 terminée\n")
    return df


def etape_7_rapport_final(controle):
    """ÉTAPE 7 : Génération du rapport final"""
    print("\n" + "="*80)
    print("  ÉTAPE 7 : RAPPORT FINAL")
    print("="*80 + "\n")
    
    controle.generer_rapport_global()
    
    print("\n" + "="*80)
    print("  ✅ TRAITEMENT TERMINÉ AVEC SUCCÈS")
    print("="*80)
    print(f"\n📁 Tous les fichiers de contrôle sont dans : {controle.output_dir}/")
    print(f"📊 {len(controle.problemes_detectes)} types de problèmes détectés au total\n")




# EXECUTION PAS A PAS  

## Chargement et description de la base 

In [12]:
# Étape 0 : Initialisation
df, controle = etape_0_initialisation()


  ÉTAPE 0 : INITIALISATION

📂 Chargement des données...
✓ Données chargées : 15,627,963 observations
✓ Nombre de matricules uniques : 275,066
✓ Période couverte : 2015-01-31 00:00:00 à 2020-12-31 00:00:00
✓ Nombre de variables : 40

  DESCRIPTION DES VARIABLES



,Variable,Type,Non-nulls,Exemple
0,MATRICULE||CODE_ORGANISME,object,"15,627,963 (100.0%)",011242X15
1,MATRICULE,object,"15,627,963 (100.0%)",011242X
2,NOM,object,"15,627,963 (100.0%)",DOSSO MEGBO
3,DATE NAISSANCE,datetime64[ns],"15,620,442 (100.0%)",1939-01-01 00:00:00
4,SEXE,object,"15,627,813 (100.0%)",Masculin
5,SITUATION MATRIMONIALE,object,"15,627,963 (100.0%)",Marié
6,NOMBRE ENFANT,float64,"15,601,334 (99.8%)",0.0
7,LIEU AFFECTATION,object,"15,627,963 (100.0%)",Seguela
8,SERVICE,object,"15,627,956 (100.0%)",D. G. A. A. T.
9,ORGANISME,object,"15,627,963 (100.0%)",Min. d'Etat Administration du Territoire



✅ Initialisation terminée



## Situation matrimoniale

In [13]:
# Étape 1 : Situation matrimoniale
df, rep_sitmat = etape_1_situation_matrimoniale(df, controle)



  ÉTAPE 1 : SITUATION MATRIMONIALE

🔄 Traitement en cours...
   → Recodage des modalités...
   → Vérification de la cohérence non temporelle...
   → Vérification de la cohérence temporelle...
   → Analyse des variations temporelles...

📊 Contrôle qualité...


  CONTRÔLE : SITUATION_MATRIMONIALE_RECODE

📊 Distribution de SITUATION_MATRIMONIALE_RECODE:


,Effectif,Pourcentage (%)
SITUATION_MATRIMONIALE_RECODE,,
Autres,25466,0.16
Célibataire,10021194,64.12
Divorcé/Séparé,12637,0.08
Marié(e),5562568,35.59
Veuf/Veuve,6098,0.04



⚠️  Changement de situation matrimoniale entre deux mois consécutifs
   Observations FLAGÉES : 57,792 (0.37%)
   Observations exportées (avec contexte) : 2,601,251
   Nombre de matricules concernés : 39,562 (14.38%)

   📋 Échantillon (10 premières lignes avec contexte) :


,MATRICULE,DATE_REF,PERIODE,DATE_COLLECTE,_SITMAT_RECODE,NOM,FLAG_SITMAT_VARIATION,DESCRIPTION_PROBLEME
184961,078222E,2015-02-01,022015,2015-02-28,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
370298,078222E,2015-03-01,032015,2015-03-31,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
556592,078222E,2015-04-01,042015,2015-04-30,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
744566,078222E,2015-05-01,052015,2015-05-31,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
934547,078222E,2015-06-01,062015,2015-06-30,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
1126891,078222E,2015-07-01,072015,2015-07-31,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
1320005,078222E,2015-08-01,082015,2015-08-31,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
1514354,078222E,2015-09-01,092015,2015-09-30,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
1709323,078222E,2015-10-01,102015,2015-10-31,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...
1905760,078222E,2015-11-01,112015,2015-11-30,celibataire,OUATTARA ZANA,0,Changement de situation matrimoniale entre deu...



   ⚠️  Fichier trop volumineux pour Excel (2,601,251 lignes)
   📦 Export en CSV et Parquet...
   💾 CSV exporté : controles_qualite/SITUATION_MATRIMONIALE_RECODE_FLAG_SITMAT_VARIATION_20251119_094311.csv
   💾 Parquet exporté : controles_qualite/SITUATION_MATRIMONIALE_RECODE_FLAG_SITMAT_VARIATION_20251119_094311.parquet

✅ Étape 1 terminée



## Nombre d'enfants

In [14]:
# Étape 2 : Nombre d'enfants
df, rep_enfants = etape_2_nombre_enfants(df, controle)


  ÉTAPE 2 : NOMBRE D'ENFANTS

🔄 Traitement en cours...
   → Imputation des valeurs manquantes...
   → Vérification de la cohérence non temporelle...
   → Vérification de la cohérence temporelle...
   → Analyse des variations temporelles...

📊 Contrôle qualité...


  CONTRÔLE : NBRE_ENFTS_IMPUTE

📊 Distribution de NBRE_ENFTS_IMPUTE:


,Effectif,Pourcentage (%)
NBRE_ENFTS_IMPUTE,,
0,5604678,35.86
1,2369618,15.16
2,2595972,16.61
3,2164280,13.85
4,1426419,9.13
5,780665,5.0
6,476700,3.05
7,133859,0.86
8,50576,0.32



⚠️  Nombre d'enfants qui diminue dans le temps → incohérence
   Observations FLAGÉES : 61,339 (0.39%)
   Observations exportées (avec contexte) : 2,641,634
   Nombre de matricules concernés : 38,521 (14.00%)

   📋 Échantillon (10 premières lignes avec contexte) :


,MATRICULE,DATE_REF,PERIODE,DATE_COLLECTE,NBRE_ENFTS_IMPUTE,NOM,FLAG_ENFANTS_DESCENTE,DESCRIPTION_PROBLEME
13,055619D,2015-01-01,012015,2015-01-31,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
184904,055619D,2015-02-01,022015,2015-02-28,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
370240,055619D,2015-03-01,032015,2015-03-31,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
556534,055619D,2015-04-01,042015,2015-04-30,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
744508,055619D,2015-05-01,052015,2015-05-31,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
934488,055619D,2015-06-01,062015,2015-06-30,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
1126832,055619D,2015-07-01,072015,2015-07-31,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
1319946,055619D,2015-08-01,082015,2015-08-31,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
1514295,055619D,2015-09-01,092015,2015-09-30,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...
1709263,055619D,2015-10-01,102015,2015-10-31,3,BAKAYOKO MOYABI,0,Nombre d'enfants qui diminue dans le temps → i...



   ⚠️  Fichier trop volumineux pour Excel (2,641,634 lignes)
   📦 Export en CSV et Parquet...
   💾 CSV exporté : controles_qualite/NBRE_ENFTS_IMPUTE_FLAG_ENFANTS_DESCENTE_20251119_094311.csv
   💾 Parquet exporté : controles_qualite/NBRE_ENFTS_IMPUTE_FLAG_ENFANTS_DESCENTE_20251119_094311.parquet

✅ Étape 2 terminée



## Sexe

In [15]:
# Étape 3 : Sexe
df = etape_3_sexe(df, controle)



  ÉTAPE 3 : SEXE

🔄 Traitement en cours...
   → Nettoyage et stabilisation...
   → Imputation des valeurs manquantes...
   → Vérification de la cohérence temporelle...
   → Analyse des variations temporelles...

📊 Contrôle qualité...


  CONTRÔLE : SEXE_IMPUTE

📊 Distribution de SEXE_IMPUTE:


,Effectif,Pourcentage (%)
SEXE_IMPUTE,,
Féminin,4684666,29.98
Masculin,10943297,70.02



✅ Aucun problème détecté pour cette variable.

✅ Étape 3 terminée



In [ ]:
Vérification 

## Age 

In [16]:
# Étape 4 : Âge
df = etape_4_age(df, controle)


  ÉTAPE 4 : ÂGE

🔄 Traitement en cours...
   → Calcul et imputation de l'âge...
   → Imputation temporellement cohérente de l'âge...


/tmp/ipykernel_154/905270905.py:137: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tdf = df.groupby(matricule_col, group_keys=False).apply(_impute_coherent)


   → Vérification de la cohérence temporelle...
   → Analyse des variations temporelles...

📊 Contrôle qualité...


  CONTRÔLE : AGE_IMPUTE

📊 Distribution de AGE_IMPUTE:


,Effectif,Pourcentage (%)
AGE_IMPUTE,,
18,79,0.0
19,340,0.0
20,1233,0.01
21,3539,0.02
22,8000,0.05
23,15790,0.1
24,29678,0.19
25,53339,0.34
26,87948,0.56



✅ Aucun problème détecté pour cette variable.

✅ Étape 4 terminée



### Vérification que les contrôles sont bien fait

In [10]:
# ============================================================
# TABLEAU AGE PAR MATRICULE ET PAR ANNÉE (2015–2020) SANS DOUBLONS
# ============================================================

# 1. S'assurer qu'on a une date au format datetime + ANNEE
if 'DATE_REF' in df.columns:
    df['DATE_REF'] = pd.to_datetime(df['DATE_REF'], errors='coerce')
    df['ANNEE'] = df['DATE_REF'].dt.year
else:
    df['DATE_COLLECTE'] = pd.to_datetime(df['DATE_COLLECTE'], errors='coerce')
    df['ANNEE'] = df['DATE_COLLECTE'].dt.year

# 2. Filtrer 2015–2020
mask = df['ANNEE'].between(2015, 2020)
tmp = df.loc[mask, ['MATRICULE', 'ANNEE', 'AGE_IMPUTE']].copy()

# 3. Supprimer les lignes strictement en doublon
tmp = tmp.drop_duplicates(subset=['MATRICULE', 'ANNEE', 'AGE_IMPUTE'])

# 4. Agréger pour garantir 1 ligne par (MATRICULE, ANNEE)
#    Ici on prend le max = âge le plus élevé observé dans l’année
tmp_agg = (
    tmp
    .groupby(['MATRICULE', 'ANNEE'], as_index=False)['AGE_IMPUTE']
    .max()
)

# 5. Pivot : lignes = MATRICULE, colonnes = ANNEE, valeurs = AGE_IMPUTE
table_age = tmp_agg.pivot(index='MATRICULE', columns='ANNEE', values='AGE_IMPUTE')

# 6. Réordonner les colonnes de 2015 à 2020
colonnes_ordre = [a for a in range(2015, 2021) if a in table_age.columns]
table_age = table_age[colonnes_ordre]

# 8. Aperçu
display(table_age.head())



ANNEE,2015,2016,2017,2018,2019,2020
MATRICULE,,,,,,
011242X,42,43,44,45,46,<NA>
012856Q,42,43,44,45,46,<NA>
013454N,42,43,44,45,46,<NA>
024732P,41,<NA>,<NA>,<NA>,<NA>,<NA>
027861L,42,43,44,<NA>,<NA>,<NA>


In [ ]:

# 9. Enregistrement dans un fichier Excel
fichier_sortie = "age_par_matricule_2015_2020_sans_doublons.xlsx"
table_age.to_excel(fichier_sortie, index=True)
print(f"✓ Fichier Excel créé : {fichier_sortie}")

### Vérification de la table de collecte 

In [10]:
# # Création du tableau avec effectifs et pourcentages (y compris NaN)
tab_annee = df["DATE_COLLECTE"].value_counts(dropna=False).sort_index()
tab_annee_pct = df["DATE_COLLECTE"].value_counts(normalize=True, dropna=False).sort_index()

# Créer le DataFrame et l’assigner à une variable
df_tab = pd.DataFrame({
    "Effectif": tab_annee,
    "Pourcentage (%)": (tab_annee_pct * 100).round(2)
})
df_tab

,Effectif,Pourcentage (%)
DATE_COLLECTE,,
2015-01-31,184890,1.18
2015-02-28,185336,1.19
2015-03-31,186294,1.19
2015-04-30,187974,1.20
2015-05-31,189980,1.22
...,...,...
2020-08-31,246523,1.58
2020-09-30,250895,1.61
2020-10-31,253185,1.62


## Âge de prise de service

In [17]:
# Étape 5 : Âge de prise de service
df = etape_5_age_service(df, controle)



  ÉTAPE 5 : ÂGE DE PRISE DE SERVICE

🔄 Traitement en cours...
   📌 MÉTHODE 2 ROBUSTE : Calcul avec détection des âges négatifs
   → Utilisation de 'AGE_IMPUTE' pour le calcul
   → Calcul de l'âge de service STABLE (version robuste avec détection des négatifs)...


/tmp/ipykernel_154/2883213670.py:173: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = df.groupby(matricule_col, group_keys=False).apply(_calc_age_service_robust)
/tmp/ipykernel_154/2883213670.py:173: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(matricule_col, group_keys=False).apply(_calc_age_service_robust)



   📊 Diagnostic des méthodes utilisées :
      ✅ PREMIERE_OBS                   : 274,339 matricules ( 99.7%)
      ✅ AUCUNE_DONNEE                  :    634 matricules (  0.2%)
      ⚠️ DERNIERE_OBS_INVALIDE          :     49 matricules (  0.0%)
      ⚠️ OBS_MEDIANE_INVALIDE           :     28 matricules (  0.0%)
      ⚠️ PREMIERE_OBS_INVALIDE          :     12 matricules (  0.0%)
      ✅ OBS_MEDIANE                    :      3 matricules (  0.0%)
      ✅ DERNIERE_OBS                   :      1 matricules (  0.0%)

   ⚠️  ATTENTION : Âges de service négatifs détectés !
      • Observations : 673 (0.00%)
      • Matricules : 20 (0.01%)

   💡 Causes possibles :
      1. Date de prise de service erronée (trop ancienne)
      2. Âge imputé trop faible
      3. Confusion entre date de naissance et date de prise de service
      4. Erreur de conversion de dates (format Excel)

   📊 Statistiques globales :
      • Observations calculées : 15,591,945 / 15,627,963 (99.8%)
      • Observations

,Effectif,Pourcentage (%)
AGE_AU_SERVICE,,
-46.905544,72,0.00
-24.245722,72,0.00
-7.577002,6,0.00
-7.334702,2,0.00
-5.585900,3,0.00
...,...,...
66.918549,59,0.00
67.053388,59,0.00
67.247775,21,0.00



⚠️  Âge de prise de service NÉGATIF → erreur grave de données
   Observations FLAGÉES : 673 (0.00%)
   Observations exportées (avec contexte) : 673
   Nombre de matricules concernés : 20 (0.01%)

   📋 Échantillon (10 premières lignes avec contexte) :


,MATRICULE,DATE_REF,PERIODE,DATE_COLLECTE,AGE_AU_SERVICE,AGE_IMPUTE,PRISE DE SERVICE,PRISE_DE_SERVICE_CORRIGEE,DATE_COLLECTE,AGE_SERVICE_METHODE,NOM,FLAG_AGE_PS_NEGATIF,DESCRIPTION_PROBLEME
0,011242X,2015-01-01,012015,2015-01-31,-4.995209,41,1969-01-01,1969-01-01,2015-01-31,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
1,011242X,2015-02-01,022015,2015-02-28,-4.995209,41,1969-01-01,1969-01-01,2015-02-28,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
2,011242X,2015-03-01,032015,2015-03-31,-4.995209,41,1969-01-01,1969-01-01,2015-03-31,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
3,011242X,2015-04-01,042015,2015-04-30,-4.995209,41,1969-01-01,1969-01-01,2015-04-30,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
4,011242X,2015-05-01,052015,2015-05-31,-4.995209,41,1969-01-01,1969-01-01,2015-05-31,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
5,011242X,2015-06-01,062015,2015-06-30,-4.995209,41,1969-01-01,1969-01-01,2015-06-30,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
6,011242X,2015-07-01,072015,2015-07-31,-4.995209,41,1969-01-01,1969-01-01,2015-07-31,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
7,011242X,2015-08-01,082015,2015-08-31,-4.995209,42,1969-01-01,1969-01-01,2015-08-31,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
8,011242X,2015-09-01,092015,2015-09-30,-4.995209,42,1969-01-01,1969-01-01,2015-09-30,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...
9,011242X,2015-10-01,102015,2015-10-31,-4.995209,42,1969-01-01,1969-01-01,2015-10-31,DERNIERE_OBS_INVALIDE,DOSSO MEGBO,1,Âge de prise de service NÉGATIF → erreur grave...


   💾 Excel exporté : controles_qualite/AGE_AU_SERVICE_FLAG_AGE_PS_NEGATIF_20251119_094311.xlsx

⚠️  Âge de prise de service >= âge actuel → impossible
   Observations FLAGÉES : 643 (0.00%)
   Observations exportées (avec contexte) : 1,840
   Nombre de matricules concernés : 28 (0.01%)

   📋 Échantillon (10 premières lignes avec contexte) :


,MATRICULE,DATE_REF,PERIODE,DATE_COLLECTE,AGE_AU_SERVICE,AGE_AU_SERVICE_VALIDE,AGE_IMPUTE,AGE,PRISE DE SERVICE,NOM,FLAG_AGE_PS_GE_AGE,DESCRIPTION_PROBLEME
2829617,243223C,2015-01-01,012015,2015-01-31,51.164271,51.164271,51,51.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829618,243223C,2015-02-01,022015,2015-02-28,51.164271,51.164271,51,51.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829619,243223C,2015-03-01,032015,2015-03-31,51.164271,51.164271,51,51.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829620,243223C,2015-04-01,042015,2015-04-30,51.164271,51.164271,51,51.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829621,243223C,2015-05-01,052015,2015-05-31,51.164271,51.164271,51,51.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829622,243223C,2015-06-01,062015,2015-06-30,51.164271,51.164271,51,51.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829623,243223C,2015-07-01,072015,2015-07-31,51.164271,51.164271,51,52.0,1991-07-25,MARIAME SOKODOGO,1,Âge de prise de service >= âge actuel → imposs...
2829624,243223C,2015-08-01,082015,2015-08-31,51.164271,51.164271,52,52.0,1991-07-25,MARIAME SOKODOGO,0,Âge de prise de service >= âge actuel → imposs...
2829625,243223C,2015-09-01,092015,2015-09-30,51.164271,51.164271,52,52.0,1991-07-25,MARIAME SOKODOGO,0,Âge de prise de service >= âge actuel → imposs...
2829626,243223C,2015-10-01,102015,2015-10-31,51.164271,51.164271,52,52.0,1991-07-25,MARIAME SOKODOGO,0,Âge de prise de service >= âge actuel → imposs...


   💾 Excel exporté : controles_qualite/AGE_AU_SERVICE_FLAG_AGE_PS_GE_AGE_20251119_094311.xlsx

✅ Étape 5 terminée



## Ancienneté

In [18]:
# Étape 6 : Ancienneté
df = etape_6_anciennete(df, controle)



  ÉTAPE 6 : ANCIENNETÉ

🔄 Traitement en cours...
   → Calcul de l'ancienneté...
   → Vérification de la cohérence non temporelle...
   → Vérification de la cohérence temporelle...
   → Analyse des variations temporelles...

📊 Contrôle qualité...


  CONTRÔLE : ANCIENNETE

📊 Distribution de ANCIENNETE:


,Effectif,Pourcentage (%)
ANCIENNETE,,
-6.67,1,0.0
-6.6,1,0.0
-6.51,1,0.0
-6.43,1,0.0
-6.35,1,0.0
...,...,...
101.65,1,0.0
101.74,1,0.0
101.82,1,0.0



⚠️  Ancienneté négative → incohérence
   Observations FLAGÉES : 558 (0.00%)
   Observations exportées (avec contexte) : 1,760
   Nombre de matricules concernés : 24 (0.01%)

   📋 Échantillon (10 premières lignes avec contexte) :


,MATRICULE,DATE_REF,PERIODE,DATE_COLLECTE,ANCIENNETE,PRISE DE SERVICE,PRISE_DE_SERVICE_CORRIGEE,NOM,FLAG_ANC_NEGATIVE,DESCRIPTION_PROBLEME
2829617,243223C,2015-01-01,012015,2015-01-31,-0.16,1991-07-25,2015-04-01,MARIAME SOKODOGO,1,Ancienneté négative → incohérence
2829618,243223C,2015-02-01,022015,2015-02-28,-0.09,1991-07-25,2015-04-01,MARIAME SOKODOGO,1,Ancienneté négative → incohérence
2829619,243223C,2015-03-01,032015,2015-03-31,-0.0,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829620,243223C,2015-04-01,042015,2015-04-30,0.08,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829621,243223C,2015-05-01,052015,2015-05-31,0.16,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829622,243223C,2015-06-01,062015,2015-06-30,0.25,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829623,243223C,2015-07-01,072015,2015-07-31,0.33,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829624,243223C,2015-08-01,082015,2015-08-31,0.42,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829625,243223C,2015-09-01,092015,2015-09-30,0.5,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence
2829626,243223C,2015-10-01,102015,2015-10-31,0.58,1991-07-25,2015-04-01,MARIAME SOKODOGO,0,Ancienneté négative → incohérence


   💾 Excel exporté : controles_qualite/ANCIENNETE_FLAG_ANC_NEGATIVE_20251119_094311.xlsx

✅ Étape 6 terminée



## Rapport final

In [19]:
# Étape 7 : Rapport final
etape_7_rapport_final(controle)


  ÉTAPE 7 : RAPPORT FINAL


📊 RAPPORT GLOBAL DES CONTRÔLES


,variable,flag,description,nb_observations,nb_matricules,pct_observations,pct_matricules,fichier
0,SITUATION_MATRIMONIALE_RECODE,FLAG_SITMAT_VARIATION,Changement de situation matrimoniale entre deu...,57792,39562,0.37,14.38,SITUATION_MATRIMONIALE_RECODE_FLAG_SITMAT_VARI...
1,NBRE_ENFTS_IMPUTE,FLAG_ENFANTS_DESCENTE,Nombre d'enfants qui diminue dans le temps → i...,61339,38521,0.39,14.00,NBRE_ENFTS_IMPUTE_FLAG_ENFANTS_DESCENTE_202511...
2,AGE_AU_SERVICE,FLAG_AGE_PS_NEGATIF,Âge de prise de service NÉGATIF → erreur grave...,673,20,0.00,0.01,AGE_AU_SERVICE_FLAG_AGE_PS_NEGATIF_20251119_09...
3,AGE_AU_SERVICE,FLAG_AGE_PS_GE_AGE,Âge de prise de service >= âge actuel → imposs...,643,28,0.00,0.01,AGE_AU_SERVICE_FLAG_AGE_PS_GE_AGE_20251119_094...
4,ANCIENNETE,FLAG_ANC_NEGATIVE,Ancienneté négative → incohérence,558,24,0.00,0.01,ANCIENNETE_FLAG_ANC_NEGATIVE_20251119_094311.xlsx



💾 Rapport global exporté vers : controles_qualite/RAPPORT_GLOBAL_CONTROLES_20251119_094311.xlsx


  ✅ TRAITEMENT TERMINÉ AVEC SUCCÈS

📁 Tous les fichiers de contrôle sont dans : controles_qualite/
📊 5 types de problèmes détectés au total

